# Invisible Warriors
13.04.2026

## Exploratory Data Analysis and Feature Engineering
Added new functions that were previously in the dataset preparation (until v5):
- feature section allocation (from external file)
- imputation of the missing values for days since october 7th
- remove meta data columns and export to a metadata df

## V2 (25.05.2026)
- outsource all of the functions to "functions.py"

In [ ]:
# ============================================================
# 1. SETUP AND CONFIGURATION
# ============================================================
# This block defines:
# - File paths
# - Output directories
# - Logging system
# - Plot saving functions
# - Global plotting style
# ============================================================

import os
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(".."))
from functions import *

# Create timestamp for reproducible versioned outputs
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")
dir_timestamp = datetime.now().strftime("%Y%m%d%H%M")

# Define paths

# Try script path first (if running as .py)
if "__file__" in globals():
    CODE_PATH = os.path.abspath(__file__)
else:
    # Fallback for notebooks
    CODE_PATH = os.path.abspath("02.EDA_and_featureengineering.ipynb")

ORIGIN_DATA_DIR = Path(r"HOME_PATH\data")

DATA_DIR = Path(r"HOME_PATH\00.Data_prep\outputs\202606031230")

# This should point to the CLEAN dataset produced by notebook 1
DATA_PATH = DATA_DIR / "analysis_ready_dataset_20260603_1230.xlsx"

BASE_DIR = Path(r"HOME_PATH\01.EDA\outputs")

SECTION_FILE = ORIGIN_DATA_DIR / "section_column_names_scales.tsv"
OUTPUT_DIR = BASE_DIR / dir_timestamp
LOG_PATH = OUTPUT_DIR / "log.txt"

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory set to:\n{OUTPUT_DIR}")

# Initialize log
if not LOG_PATH.exists():
    write_log(f"Session timestamp: {TIMESTAMP}", LOG_PATH)
    write_log(f"Running EDA on dataset: {DATA_PATH}",LOG_PATH)
    write_log("EDA notebook started.",LOG_PATH)
    write_log(f"Output directory set to: {OUTPUT_DIR}",LOG_PATH)

# ============================================================
# Global plotting style
# ============================================================
sns.set_theme(style="whitegrid",context="talk")
plt.rcParams["figure.figsize"] = (8,6)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["figure.dpi"] = 120
write_log("Plotting style configured.", LOG_PATH)

In [ ]:
# ============================================================
# 2. LOAD CLEAN DATASET
# ============================================================
# Load the dataset produced by the data preparation notebook.
# This dataset should contain:
# - cleaned column names
# - removed metadata columns
# - removed derived scores
# - participant QC already applied
# ============================================================
write_log("Loading clean dataset.", LOG_PATH)

df = pd.read_excel(DATA_PATH)

write_log(f"Dataset loaded successfully.", LOG_PATH)
write_log(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns", LOG_PATH)

print("\nDataset preview:")
display(df.head())

In [ ]:
# =======================================================================
# 2.1 Build subsection + section dictionary from section file
# (schema-preserving, auto-complete unmapped features)
# =======================================================================
write_log("Loading subsection definitions.", LOG_PATH)

SECTION_FILE = DATA_DIR / "section_column_names_scales.tsv"
sections_df = pd.read_csv(SECTION_FILE, sep="\t")

write_log(f"Section definition file loaded. Rows: {len(sections_df)}", LOG_PATH)
display(sections_df)

# Keep only rows whose feature exists in df
# This preserves the original schema exactly as defined in SECTION_FILE
# while dropping stale mappings for features no longer present in df
write_log("Filtering section definitions to features present in df.", LOG_PATH)

sections_df = sections_df.loc[sections_df["column_name"].isin(df.columns)].copy()

write_log(f"Valid section rows retained: {len(sections_df)}", LOG_PATH)

# Auto-add unmapped df features
# Any feature present in df but absent from the section file is appended
# as section='unmapped', subsection='unmapped', while preserving the full
# original schema of sections_df (all other metadata columns stay intact).
mapped_cols = set(sections_df["column_name"])
unmapped_cols = [col for col in df.columns if col not in mapped_cols]

if unmapped_cols:
    unmapped_rows = pd.DataFrame(columns=sections_df.columns)
    unmapped_rows["column_name"] = unmapped_cols
    unmapped_rows["section"] = "unmapped"
    unmapped_rows["subsection"] = "unmapped"

    sections_df = pd.concat([sections_df, unmapped_rows], ignore_index=True)

write_log(f"Unmapped features auto-added: {len(unmapped_cols)}", LOG_PATH)

# Validate uniqueness
# After auto-completion, every df feature should appear exactly once in
# sections_df. Duplicate assignments indicate a broken section file.
dup_count = sections_df["column_name"].duplicated().sum()
if dup_count > 0:
    dup_cols = sections_df.loc[
        sections_df["column_name"].duplicated(), "column_name"
    ].tolist()
    raise ValueError(
        f"Duplicate column assignments across subsections: {dup_count} "
        f"({', '.join(dup_cols)})"
    )

write_log("Subsection uniqueness validation passed.", LOG_PATH)

# Build dictionaries from canonical section table
# sections_df is now the complete source of truth for all df features
subsection_dict = (
    sections_df
    .groupby("subsection", sort=False)["column_name"]
    .apply(list)
    .to_dict()
)

section_dict = (
    sections_df
    .groupby("section", sort=False)["column_name"]
    .apply(list)
    .to_dict()
)

write_log("Subsection and section dictionaries created.", LOG_PATH)
write_log(f"Total subsections: {len(subsection_dict)}", LOG_PATH)
write_log(f"Total sections: {len(section_dict)}", LOG_PATH)

for k, v in subsection_dict.items():
    write_log(f"Subsection {k}: {len(v)} columns", LOG_PATH)

for k, v in section_dict.items():
    write_log(f"Section {k}: {len(v)} columns", LOG_PATH)

# Export long-format mapping
# sections_df is already long-format (one row per feature), so export a
# schema-preserving copy and add helper columns only at export time
long_df = sections_df.copy()

# Add subsection size as a convenience field for downstream inspection.
long_df["n_columns"] = long_df.groupby("subsection")["column_name"].transform("count")

long_df.to_csv(OUTPUT_DIR / "section_column_names_scales.tsv",sep="\t",index=False)

write_log("Long-format subsection-column mapping exported.", LOG_PATH)

# Subsection audit summary
subsection_audit = (
    sections_df
    .groupby(["subsection", "section"], sort=False)
    .agg(
        n_columns=("column_name", "count"),
        column_names=("column_name", ", ".join)
    )
    .reset_index()
)

subsection_audit.to_csv(OUTPUT_DIR / "subsection_dictionary_summary.csv",index=False)

write_log("Subsection dictionary summary exported.", LOG_PATH)

# Section audit summary
section_audit = (
    sections_df
    .groupby("section", sort=False)
    .agg(
        n_columns=("column_name", "count"),
        column_names=("column_name", ", ".join)
    )
    .reset_index()
)

section_audit.to_csv(OUTPUT_DIR / "section_dictionary_summary.csv",index=False)

write_log("Section dictionary summary exported.", LOG_PATH)

# Reconciliation summary
# After auto-completion, df_not_in_dict should always be zero unless
# something failed upstream
dict_cols = sorted(sections_df["column_name"].unique().tolist())
df_cols = sorted(df.columns.tolist())

dict_not_in_df = sorted(set(dict_cols) - set(df_cols))
df_not_in_dict = sorted(set(df_cols) - set(dict_cols))

write_log("Column reconciliation summary:", LOG_PATH)
write_log(f"Columns in dictionary but missing from df: {len(dict_not_in_df)}", LOG_PATH)

if dict_not_in_df:
    write_log("Missing from df: " + ", ".join(dict_not_in_df), LOG_PATH)

write_log(f"Columns in df but missing from dictionary: {len(df_not_in_dict)}", LOG_PATH)

if df_not_in_dict:
    write_log("Missing from dictionary: " + ", ".join(df_not_in_dict), LOG_PATH)

In [ ]:
# ============================================================
# GLOBAL FEATURE REMOVAL AUDIT OBJECTS
# ============================================================
# Output Excel path
FEATURE_REMOVAL_AUDIT_PATH = (OUTPUT_DIR / "feature_removal_audit.xlsx")
FEATURE_REMOVAL_LOG_PATH = (OUTPUT_DIR / "feature_removal_log.xlsx")

# FEATURE REMOVAL REGISTRY
feature_removal_registry = []

# Initialize in-memory audit tables
# IMPORTANT: These persist across notebook execution

if "removed_feature_values_df" not in globals():
    removed_feature_values_df = pd.DataFrame()

if "removal_log_df" not in globals():
    removal_log_df = pd.DataFrame()

In [ ]:
# Clean Meta Columns from the Main Dataset

# Separate the technical columns from the main DF - COME BACK TO THIS!! THE SECTIONS HAVE CHANGED
write_log("Starting separation of technical / non-useful REDCap columns.", LOG_PATH)

# 1. Automatically detect timestamp columns (case-insensitive)
timestamp_columns = [col for col in df.columns if "timestamp" in col.lower()]

if timestamp_columns:
    write_log(f"*_timestamp columns marked for removal: {len(timestamp_columns)} \n{timestamp_columns}", LOG_PATH)

# 2. Automatically detect section completion columns
complete_columns = [col for col in df.columns if "complete" in col.lower()]

if complete_columns:
    write_log(f"*_completed columns marked for removal: {len(complete_columns)} \n{complete_columns}", LOG_PATH)

# 3. Remove entire questionnaire sections that are not relevant for analysis (using the section dictionary)
sections_to_remove = [
    "attestation",
    "partners_of_reservists_study"
]

section_columns_to_remove = []

for section in sections_to_remove:
    if section in section_dict:
        section_columns_to_remove.extend(section_dict[section])

write_log(f"Columns of excluded section marked for removal: {len(section_columns_to_remove)} \n{section_columns_to_remove}", LOG_PATH)

# 5. Remove derived / calculated variables that will be recomputed later in the analysis pipeline
derived_columns_to_remove = [
    "pcl5_c",
    "pcl5_cevillian_cutoff",
    "ptg_i_total",
    "ptg_i_categorization",
    "ptg_relating_others",
    "ptg_new_possibilities",
    "ptg_personal_strength",
    "ptg_spiritual",
    "ptg_life_appreciation"
]

write_log(f"Derived variables marked for removal: {len(derived_columns_to_remove)} \n{derived_columns_to_remove}", LOG_PATH)

# 6. Remove variables that were deemed uninformative (manually)

general_columns_to_remove = [
    "rela_dur_v2",
    "married_v2",
    "sex_v2", # due to an encoding problem. can assume all are female because of the way data was collected
    "sex_partner_v2" # due to an encoding problem
]

write_log(f"General other variables marked for removal: {len(general_columns_to_remove)} \n{general_columns_to_remove}", LOG_PATH)

# 7. Audit and mark for removal features from the main df, and keep track
feature_removal_registry.append({
    "columns": [section_columns_to_remove, complete_columns, timestamp_columns],
    "removal_reason": "Technical metadata columns",
    "removal_stage": "data_cleaning"
})

feature_removal_registry.append({
    "columns": derived_columns_to_remove,
    "removal_reason": "Derived features, will be recreated in code",
    "removal_stage": "data_cleaning"
})

feature_removal_registry.append({
    "columns": general_columns_to_remove,
    "removal_reason": "Logical redundancy",
    "removal_stage": "data_cleaning"
})

In [ ]:
# ============================================================
# 3. DATASET STRUCTURAL DIAGNOSTICS
# ============================================================
# This block explores the structural properties of the dataset.
# ============================================================
write_log("Running dataset structural diagnostics.", LOG_PATH)

# Basic dataset info
n_rows, n_cols = df.shape
write_log(f"Number of participants: {n_rows}", LOG_PATH)
write_log(f"Number of variables: {n_cols}", LOG_PATH)
print("\nDataset shape:")
print(df.shape)
print("\nColumn type counts:")
display(df.dtypes.value_counts())

# Detect REDCap checkbox variables
# REDCap encodes checkboxes as: variable___1, variable___2, variable___3
write_log("Detecting checkbox variables.", LOG_PATH)
checkbox_columns = [c for c in df.columns if "___" in c]
checkbox_groups = {}

for col in checkbox_columns:
    base = col.split("___")[0]
    if base not in checkbox_groups:
        checkbox_groups[base] = []
    checkbox_groups[base].append(col)

write_log(f"Detected {len(checkbox_groups)} checkbox groups.", LOG_PATH)
print("\nCheckbox groups detected:")

for base, cols in checkbox_groups.items():
    print(f"{base}: {len(cols)} options")

# Handle Missing Data


In [ ]:
# ============================================================
# MISSING DATA SUMMARY — fraction missing by column
# ============================================================
write_log("Computing missing data summary by column.", LOG_PATH)

# ------------------------------------------------------------
# Step 1 — Compute fraction missing per column
# ------------------------------------------------------------
# Only true missing values are counted (NaN / None).
# Numeric 0 is treated as a valid observed value.
missing_frac = df.isna().mean() 

# ------------------------------------------------------------
# Step 2 — Sort descending by missingness
# ------------------------------------------------------------
missing_summary = (missing_frac.sort_values(ascending=False).reset_index())
missing_summary.columns = ["column_name", "frac_missing"]

# ------------------------------------------------------------
# Step 3 — Display result
# ------------------------------------------------------------
display(missing_summary)
missing_summary.plot()

In [ ]:
# ============================================================
# IDENTIFY + AUDIT ALL MISSING FEATURES
# ============================================================
MISSINGNESS_THRESHOLD = 0.99
# ------------------------------------------------------------
# Get columns with 100% missingness
# Reuses existing missing_summary table
# ------------------------------------------------------------
fully_missing_cols = (missing_summary.loc[missing_summary["frac_missing"] >= MISSINGNESS_THRESHOLD,"column_name"].tolist())
# mark for removal
feature_removal_registry.append({
    "columns": fully_missing_cols,
    "removal_reason": (f"missing > {MISSINGNESS_THRESHOLD}"),
    "removal_stage": "handle_missing_data"
})

write_log(f"Features with missingness over {MISSINGNESS_THRESHOLD}: {len(fully_missing_cols)}", LOG_PATH)

In [ ]:
# ============================================================
# Participant-level completeness - 2 plots
# ============================================================
write_log("Calculating participant completeness.", LOG_PATH)

# Count non-missing responses per participant (excluding record_id)
df["n_answered"] = df.drop(columns=["record_id"]).notna().sum(axis=1)

# Percentage completeness
df["pct_answered"] = (df["n_answered"] / (df.shape[1] - 1) * 100)

write_log(
    f"Participant completeness calculated. "
    f"Mean={df['pct_answered'].mean():.2f}%, "
    f"Median={df['pct_answered'].median():.2f}%, "
    f"Min={df['pct_answered'].min():.2f}%.",
    LOG_PATH
)

# ============================================================
# Multi-panel figure: survey completeness overview
# Left: Distribution histogram
# Right: Completion percentage per participant
# ============================================================
write_log("Creating multi-panel survey completeness figure.", LOG_PATH)

# Sort dataframe for consistent visualization
df_sorted = df.sort_values("record_id")

# Create figure with 2 subplots side-by-side
fig, axes = plt.subplots(nrows=1,ncols=2,figsize=(16, 6))
ax1, ax2 = axes

# Left panel — histogram

sns.histplot(df["pct_answered"],bins=20,kde=True,ax=ax1)
median_val = df["pct_answered"].median()

ax1.axvline(median_val,linestyle="--")
ax1.set_title("Distribution of Survey Completion")
ax1.set_xlabel("Percentage of Questions Answered")
ax1.set_ylabel("Number of Participants")
ax1.text(median_val,ax1.get_ylim()[1]*0.9,f"Median = {median_val:.1f}%",rotation=90,verticalalignment="top")

# Right panel — scatter plot per participant
ax2.scatter(df_sorted["record_id"],df_sorted["pct_answered"],)

ax2.axhline(median_val,linestyle="--")

ax2.set_title("Survey Completion per Participant")
ax2.set_xlabel("Participant ID (record_id)")
ax2.set_ylabel("Percentage of Questions Answered")
ax2.text(df_sorted["record_id"].min(),median_val + 1,f"Median = {median_val:.1f}%")

# Final formatting and save
plt.tight_layout()
save_plot(fig, "survey_completeness_overview", OUTPUT_DIR, LOG_PATH)
plt.show()
plt.close(fig)

write_log("Multi-panel survey completeness figure created and saved.", LOG_PATH)

In [ ]:
# after having used the timestamp columns to fix the days since october 7th, i can remove them
feature_removal_registry.append({
    "columns": timestamp_columns,
    "removal_reason": "Technical timestamp columns",
    "removal_stage": "data cleaning"
})

In [ ]:
## IMPORTANT TO DO!!! impute missing values for media consumption, using the rules i thought of with avital, on the original column values (and into the "clean" column).

# Feature Engineering

In [ ]:
# ========================================
# ADD FEATURE — FRACTION OF TIME IN RESERVE
# ========================================
# create a new feature fraction_in_reserve; it is the total days in reserve duty. relative to the total days since the war began (per participant)
# the new feature inherits the section and subsection of reserv_days

write_log("Add feature: fraction of time in reserve duty relative to days since October 7.", LOG_PATH)

# ------------------------------------------------------------
# Step 1 — Validate required columns exist
# ------------------------------------------------------------
required_cols = ["reserv_days", "days_since_oct7"]

missing_required = [col for col in required_cols if col not in df.columns]

if missing_required:
    raise ValueError(f"Cannot compute fraction_in_reserve. Missing required columns: {missing_required}")

# ------------------------------------------------------------
# Step 2 — Ensure numeric inputs
# ------------------------------------------------------------
# Coerce to numeric in case columns contain strings / imported artifacts
reserv_days_num = pd.to_numeric(df["reserv_days"], errors="coerce")
days_since_num = pd.to_numeric(df["days_since_oct7"], errors="coerce")

# ------------------------------------------------------------
# Step 3 — Compute fraction of time in reserve
# ------------------------------------------------------------
# Definition:
# fraction_in_reserve = total reserve days / total days since Oct 7

# Invalid cases remain missing:
# - missing numerator
# - missing denominator
# - denominator <= 0
df["fraction_in_reserve"] = np.where(
    days_since_num > 0,
    reserv_days_num / days_since_num,
    np.nan
)

# ------------------------------------------------------------
# Step 4 — QC flags
# ------------------------------------------------------------
# Missing = could not compute
df["qc_fraction_in_reserve_missing"] = df["fraction_in_reserve"].isna()

# Negative values should never occur
df["qc_fraction_in_reserve_negative"] = (
    df["fraction_in_reserve"].notna() &
    (df["fraction_in_reserve"] < 0)
)

# Values > 1 are logically suspicious
# (participant reports more reserve days than total elapsed war days)
df["qc_fraction_in_reserve_gt1"] = (
    df["fraction_in_reserve"].notna() &
    (df["fraction_in_reserve"] > 1)
)

# ------------------------------------------------------------
# Step 5 — Logging
# ------------------------------------------------------------
total = len(df)

n_missing = df["qc_fraction_in_reserve_missing"].sum()
n_negative = df["qc_fraction_in_reserve_negative"].sum()
n_gt1 = df["qc_fraction_in_reserve_gt1"].sum()

write_log(f"fraction_in_reserve — missing: {n_missing} ({n_missing/total:.2%})", LOG_PATH)
write_log(f"fraction_in_reserve — negative: {n_negative} ({n_negative/total:.2%})", LOG_PATH)
write_log(f"fraction_in_reserve — >1: {n_gt1} ({n_gt1/total:.2%})", LOG_PATH)

# ------------------------------------------------------------
# Step 6 — Inherit dictionary labels from reserv_days
# ------------------------------------------------------------
# Use reserv_days as the source label and register the new feature
# across all dictionary objects:
# 1. subsection_dict
# 2. section_dict
# 3. subsection_column_names_scales

# this is a patch for compatibility
subsection_column_names_scales = sections_df.copy()

source_col = "reserv_days"
new_col = "fraction_in_reserve"

if source_col in subsection_column_names_scales["column_name"].values:

    # --------------------------------------------------------
    # Look up source labels from existing dictionary table
    # --------------------------------------------------------
    source_row = subsection_column_names_scales.loc[subsection_column_names_scales["column_name"] == source_col].iloc[0]
    source_subsection = source_row["subsection"]
    source_section = source_row["section"]

    # --------------------------------------------------------
    # Register new feature across all dictionary structures
    # --------------------------------------------------------
    update_column_label(
        column_name=new_col,
        subsection=source_subsection,
        section=source_section,
        subsection_dict=subsection_dict,
        section_dict=section_dict,
        subsection_column_names_scales=subsection_column_names_scales,
        log_path=LOG_PATH
    )

    write_log(f"Dictionary inheritance assigned: {new_col} -> {source_col}", LOG_PATH)

else:
    write_log(
        f"WARNING: Could not assign dictionary inheritance for {new_col} "
        f"(source column '{source_col}' not found in dictionary).", 
        LOG_PATH
    )

get_column_label("fraction_in_reserve", subsection_dict = subsection_dict, section_dict = section_dict)

In [ ]:
# ========================================
# PLOT — DISTRIBUTION OF fraction_in_reserve
# ========================================
write_log("Plotting distribution of fraction_in_reserve.", LOG_PATH)

# ------------------------------------------------------------
# Step 1 — Coerce to numeric and isolate valid plotting range
# ------------------------------------------------------------
# Keep only numeric values and exclude missing
plot_data = pd.to_numeric(df["fraction_in_reserve"].round(2),errors="coerce").dropna()

# Identify values outside expected fraction range
out_of_range_mask = (
    (plot_data < 0) |
    (plot_data > 1)
)

out_of_range_values = plot_data[out_of_range_mask]
n_total = len(plot_data)
n_out_of_range = len(out_of_range_values)

# Log suspicious values and exclude from plotting
if n_out_of_range > 0:
    write_log(
        f"fraction_in_reserve values outside expected "
        f"range [0, 1]: "
        f"{n_out_of_range} "
        f"({n_out_of_range / n_total:.2%})"
    )

    write_log("Out-of-range values: ".join([f"{x:.4f}" for x in sorted(out_of_range_values.unique())]))

plot_data = plot_data[~out_of_range_mask]

# ------------------------------------------------------------
# Step 2 — Compute distribution statistics
# ------------------------------------------------------------
dist_stats = {
    "N": len(plot_data),
    "Mean": plot_data.mean(),
    "Median": plot_data.median(),
    "SD": plot_data.std(),
    "Min": plot_data.min(),
    "Max": plot_data.max(),
    "Q1": plot_data.quantile(0.25),
    "Q3": plot_data.quantile(0.75)
}

# ------------------------------------------------------------
# Log statistics
# ------------------------------------------------------------
write_log("fraction_in_reserve distribution statistics:", LOG_PATH)

for k, v in dist_stats.items():

    if isinstance(v, (float, np.floating)):

        write_log(f"{k}: {v:.4f}", LOG_PATH)

    else:

        write_log(f"{k}: {v}", LOG_PATH)

# ------------------------------------------------------------
# Step 3 — Plot histogram + KDE
# ------------------------------------------------------------
# X-axis fixed to [0, 1]
# because this is a fraction by definition
fig, ax = plt.subplots(figsize=(8, 5))

plot_data.plot(
    kind="hist",
    bins=30,
    density=True,
    alpha=0.7,
    ax=ax
)

plot_data.plot(
    kind="kde",
    ax=ax
)

# ------------------------------------------------------------
# Axis formatting
# ------------------------------------------------------------
ax.set_xlim(0, 1)

ax.set_xlabel("Fraction of time in reserve")
ax.set_ylabel("Density")

ax.set_title(
    "Distribution of Fraction in Reserve"
)

# ------------------------------------------------------------
# Add summary statistics onto figure
# ------------------------------------------------------------
stats_text = (
    f"N = {dist_stats['N']}\n"
    f"Mean = {dist_stats['Mean']:.3f}\n"
    f"Median = {dist_stats['Median']:.3f}\n"
    f"SD = {dist_stats['SD']:.3f}\n"
    f"Min = {dist_stats['Min']:.3f}\n"
    f"Max = {dist_stats['Max']:.3f}\n"
    f"Q1 = {dist_stats['Q1']:.3f}\n"
    f"Q3 = {dist_stats['Q3']:.3f}"
)

ax.text(0.98,0.98,stats_text,transform=ax.transAxes,ha="right",va="top",fontsize=10,bbox=dict(facecolor="white",alpha=0.85,edgecolor="gray"))

save_plot(
    fig,
    "fraction_in_reserve_distribution",
    output_dir=OUTPUT_DIR,
    log_path=LOG_PATH
)
plt.show()
plt.close()

In [ ]:
# create features of section scores 
# give clear names for each derived feature, so that can be easily separated
# check which derived features were in the original df, and re-create them (look for formulas!)
# for each one: 
#    1. check the correlation of the score components 
#    2. decide what to do with partial answering

## Scale Scores


### create a function:
input:
- feature names
- scale (min and max)
- new feature name

process: 
1. checks that the scale is as indicated. if not - fix in new columns with _scalefix suffix.
2. check the score components "speak the same language"
3. sum input features (if features altered, use _scalefix features)
4. place score into new feature_name
5. remove the score components and place them in the meta df

In [ ]:
# ============================================================
# Convert all coping_strategy_* subsections
# from [0, 4] to [1, 5]
# ============================================================

target_section = "coping_strategy"
target_prefix = "coping_strategy_"

# ------------------------------------------------------------
# Step 1 — Identify matching subsections from dictionary
# ------------------------------------------------------------
target_subsections = [
    subsection
    for subsection in subsection_dict.keys()
    if subsection.startswith(target_prefix)
]

# Keep only subsections that actually belong to the requested section
target_subsections = [
    subsection
    for subsection in target_subsections
    if get_column_label(subsection_dict[subsection][0], subsection_dict, section_dict)["section"] == target_section
]

write_log(
    f"Found {len(target_subsections)} matching subsections in section '{target_section}': "
    + ", ".join(target_subsections),
    LOG_PATH
)

# ------------------------------------------------------------
# Step 2 — Apply scale conversion to each matching subsection
# ------------------------------------------------------------
for subsection in target_subsections:

    write_log(f"Running scale conversion for subsection: {subsection}", LOG_PATH)

    convert_section_subsection_scale(
        section=target_section,
        subsection=subsection,
        current_scale=[0, 4],
        new_scale=[1, 5],
        dataframe=df,
        log_path=LOG_PATH,
        subsection_dict=subsection_dict,
        section_dict=section_dict,
        subsection_column_names_scales=subsection_column_names_scales
    )

In [ ]:
# ============================================================
# VALUE SUMMARY — original + _scalefix features
# one row per feature
# ============================================================

# ------------------------------------------------------------
# Step 1 — Find matching subsections
# ------------------------------------------------------------
matching_subsections = [
    sub for sub in subsection_dict.keys()
    if sub.startswith(target_prefix)
]

if len(matching_subsections) == 0:
    raise ValueError(f"No subsections found matching '{target_prefix}*'.")

matching_subsections = [
    sub for sub in matching_subsections
    if len(subsection_dict[sub]) > 0
    and get_column_label(subsection_dict[sub][0], subsection_dict, section_dict)["section"] == target_section
]

# ------------------------------------------------------------
# Step 2 — Build one summary row per feature
# ------------------------------------------------------------
summary_rows = []

for subsection in matching_subsections:

    # Pull all dataframe columns registered to this subsection
    subsection_features = [
        col for col in df.columns
        if col in subsection_dict[subsection]
    ]

    for col in subsection_features:

        values = pd.to_numeric(df[col], errors="coerce")

        summary_rows.append({
            "section": section,
            "subsection": subsection,
            "feature": col,
            "feature_type": "scalefix" if col.endswith("_scalefix") else "original",
            "observed_min": values.min(skipna=True),
            "observed_max": values.max(skipna=True),
            "n_missing": values.isna().sum(),
            "pct_missing": values.isna().mean() * 100,
            "value_counts": dict(
                df[col]
                .value_counts(dropna=False)
                .sort_index()
            )
        })

# ------------------------------------------------------------
# Step 3 — Output summary dataframe
# ------------------------------------------------------------
summary_df = pd.DataFrame(summary_rows)

summary_df = summary_df.sort_values(
    by=["subsection", "feature_type", "feature"]
).reset_index(drop=True)

display(summary_df)

In [ ]:
# ============================================================
# REPLACE ORIGINAL VALUES WITH _scalefix VALUES
# overwrite original features, then remove *_scalefix columns
# ============================================================

write_log("Replacing original values with _scalefix values.", LOG_PATH)

# ------------------------------------------------------------
# Step 1 — Identify all *_scalefix columns directly from df
# ------------------------------------------------------------
# Use raw dataframe columns as source of truth.
# This avoids dictionary-state issues and guarantees replacement
# and dropping operate on the exact same set of columns.
scalefix_cols = [
    col for col in df.columns
    if col.endswith("_scalefix")
]

if len(scalefix_cols) == 0:
    write_log("No _scalefix columns found. Nothing to replace.", LOG_PATH)

else:
    # --------------------------------------------------------
    # Step 2 — Overwrite original values from *_scalefix copies
    # --------------------------------------------------------
    replaced_features = []

    for scalefix_col in scalefix_cols:

        original_col = scalefix_col.replace("_scalefix", "")

        # Safety check: original column must exist
        if original_col not in df.columns:
            write_log(
                f"WARNING: original column missing, skipping replacement for '{scalefix_col}'",
                LOG_PATH
            )
            continue

        # Overwrite original values with corrected values
        df[original_col] = df[scalefix_col]

        replaced_features.append(original_col)

        write_log(f"Replaced values: {original_col} <- {scalefix_col}", LOG_PATH)

    # --------------------------------------------------------
    # Step 3 — Drop *_scalefix columns from dataframe
    # --------------------------------------------------------
    df.drop(columns=scalefix_cols, inplace=True)
    write_log(f"Dropped {len(scalefix_cols)} _scalefix columns from dataframe.", LOG_PATH)

    # --------------------------------------------------------
    # Step 4 — Remove *_scalefix columns from dictionaries
    # --------------------------------------------------------
    for subsection in subsection_dict:
        subsection_dict[subsection] = [
            c for c in subsection_dict[subsection]
            if not c.endswith("_scalefix")
        ]

    for section in section_dict:
        section_dict[section] = [
            c for c in section_dict[section]
            if not c.endswith("_scalefix")
        ]

    subsection_column_names_scales = subsection_column_names_scales[
        ~subsection_column_names_scales["column_name"].str.endswith("_scalefix")
    ].reset_index(drop=True)

    # --------------------------------------------------------
    # Step 5 — Final logging
    # --------------------------------------------------------
    write_log(
        f"Replacement complete | overwritten {len(replaced_features)} original features.",
        LOG_PATH
    )

In [ ]:
# ============================================================
# Convert all ptsd_* subsections
# from [0, 4] to [1, 5]
# ============================================================

target_section = "ptsd"
target_prefix = "ptsd_"

# ------------------------------------------------------------
# Step 1 — Identify matching subsections from dictionary
# ------------------------------------------------------------
target_subsections = [
    subsection
    for subsection in subsection_dict.keys()
    if subsection.startswith(target_prefix)
]

# Keep only subsections that actually belong to the requested section
target_subsections = [
    subsection
    for subsection in target_subsections
    if get_column_label(subsection_dict[subsection][0], subsection_dict, section_dict)["section"] == target_section
]

write_log(
    f"Found {len(target_subsections)} matching subsections in section '{target_section}': "
    + ", ".join(target_subsections),
    LOG_PATH
)

# ------------------------------------------------------------
# Step 2 — Apply scale conversion to each matching subsection
# ------------------------------------------------------------
for subsection in target_subsections:

    write_log(f"Running scale conversion for subsection: {subsection}", LOG_PATH)

    convert_section_subsection_scale(
        section=target_section,
        subsection=subsection,
        current_scale=[0, 4],
        new_scale=[1, 5],
        dataframe=df,
        log_path=LOG_PATH,
        subsection_dict=subsection_dict,
        section_dict=section_dict,
        subsection_column_names_scales=subsection_column_names_scales
    )

In [ ]:
# ============================================================
# VALUE SUMMARY — original + _scalefix features
# one row per feature
# ============================================================

# ------------------------------------------------------------
# Step 1 — Find matching subsections
# ------------------------------------------------------------
matching_subsections = [
    sub for sub in subsection_dict.keys()
    if sub.startswith(target_prefix)
]

if len(matching_subsections) == 0:
    raise ValueError(f"No subsections found matching '{target_prefix}*'.")

matching_subsections = [
    sub for sub in matching_subsections
    if len(subsection_dict[sub]) > 0
    and get_column_label(subsection_dict[sub][0], subsection_dict, section_dict)["section"] == target_section
]

# ------------------------------------------------------------
# Step 2 — Build one summary row per feature
# ------------------------------------------------------------
summary_rows = []

for subsection in matching_subsections:

    # Pull all dataframe columns registered to this subsection
    subsection_features = [
        col for col in df.columns
        if col in subsection_dict[subsection]
    ]

    for col in subsection_features:

        values = pd.to_numeric(df[col], errors="coerce")

        summary_rows.append({
            "section": section,
            "subsection": subsection,
            "feature": col,
            "feature_type": "scalefix" if col.endswith("_scalefix") else "original",
            "observed_min": values.min(skipna=True),
            "observed_max": values.max(skipna=True),
            "n_missing": values.isna().sum(),
            "pct_missing": values.isna().mean() * 100,
            "value_counts": dict(
                df[col]
                .value_counts(dropna=False)
                .sort_index()
            )
        })

# ------------------------------------------------------------
# Step 3 — Output summary dataframe
# ------------------------------------------------------------
summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(by=["subsection", "feature_type", "feature"]).reset_index(drop=True)
#display(summary_df)

In [ ]:
# ============================================================
# REPLACE ORIGINAL VALUES WITH _scalefix VALUES
# overwrite original features, then remove *_scalefix columns
# ============================================================
write_log("Replacing original values with _scalefix values.", LOG_PATH)

# ------------------------------------------------------------
# Step 1 — Identify all *_scalefix columns directly from df
# ------------------------------------------------------------
# Use raw dataframe columns as source of truth.
# This avoids dictionary-state issues and guarantees replacement
# and dropping operate on the exact same set of columns.
scalefix_cols = [
    col for col in df.columns
    if col.endswith("_scalefix")
]

if len(scalefix_cols) == 0:
    write_log("No _scalefix columns found. Nothing to replace.", LOG_PATH)

else:
    # --------------------------------------------------------
    # Step 2 — Overwrite original values from *_scalefix copies
    # --------------------------------------------------------
    replaced_features = []

    for scalefix_col in scalefix_cols:

        original_col = scalefix_col.replace("_scalefix", "")

        # Safety check: original column must exist
        if original_col not in df.columns:
            write_log(
                f"WARNING: original column missing, skipping replacement for '{scalefix_col}'", 
                LOG_PATH
            )
            continue

        # Overwrite original values with corrected values
        df[original_col] = df[scalefix_col]

        replaced_features.append(original_col)

        write_log(f"Replaced values: {original_col} <- {scalefix_col}", LOG_PATH)

    # --------------------------------------------------------
    # Step 3 — Drop *_scalefix columns from dataframe
    # --------------------------------------------------------
    df.drop(columns=scalefix_cols, inplace=True)
    write_log(f"Dropped {len(scalefix_cols)} _scalefix columns from dataframe.", LOG_PATH)

    # --------------------------------------------------------
    # Step 4 — Remove *_scalefix columns from dictionaries
    # --------------------------------------------------------
    for subsection in subsection_dict:
        subsection_dict[subsection] = [
            c for c in subsection_dict[subsection]
            if not c.endswith("_scalefix")
        ]

    for section in section_dict:
        section_dict[section] = [
            c for c in section_dict[section]
            if not c.endswith("_scalefix")
        ]

    subsection_column_names_scales = subsection_column_names_scales[
        ~subsection_column_names_scales["column_name"].str.endswith("_scalefix")
    ].reset_index(drop=True)

    # --------------------------------------------------------
    # Step 5 — Final logging
    # --------------------------------------------------------
    write_log(
        f"Replacement complete | overwritten {len(replaced_features)} original features.", 
        LOG_PATH
    )

In [ ]:
# ============================================================
# REVERSE SPECIFIC FEATURES
# ============================================================

features_to_reverse = [
    "go_wrong",
    "work_out_myway",
    "good_things"
]

scale_min = 0
scale_max = 4

write_log("Reversing selected features.", LOG_PATH)

for feature in features_to_reverse:

    # --------------------------------------------------------
    # Step 1 — Create reversed feature
    # --------------------------------------------------------
    reversed_feature = reverse_scale(
        feature_name=feature,
        scale_min=scale_min,
        scale_max=scale_max,
        dataframe=df, 
        log_path=LOG_PATH
    )

    # --------------------------------------------------------
    # Step 2 — Inherit dictionary labels from source feature
    # --------------------------------------------------------
    source_label = get_column_label(feature, subsection_dict=subsection_dict, section_dict=section_dict)

    update_column_label(
        column_name=reversed_feature,
        subsection=source_label["subsection"],
        section=source_label["section"],
        subsection_dict=subsection_dict,
        section_dict=section_dict,
        subsection_column_names_scales=subsection_column_names_scales,
        log_path=LOG_PATH
    )

write_log("Selected feature reversal complete.", LOG_PATH)

In [ ]:
# ============================================================
# VALUE SUMMARY — original + _rev features
# one row per feature
# ============================================================

features_to_check = features_to_reverse

# ------------------------------------------------------------
# Step 1 — Build feature list (original + reversed)
# ------------------------------------------------------------
summary_features = []

for feature in features_to_check:

    # Original feature
    if feature in df.columns:
        summary_features.append(feature)

    # Reversed feature
    rev_feature = f"{feature}_rev"
    if rev_feature in df.columns:
        summary_features.append(rev_feature)

# Remove duplicates while preserving order
summary_features = list(dict.fromkeys(summary_features))

if len(summary_features) == 0:
    raise ValueError("No matching original / _rev features found in dataframe.")

# ------------------------------------------------------------
# Step 2 — Build one summary row per feature
# ------------------------------------------------------------
summary_rows = []

for col in summary_features:

    values = pd.to_numeric(df[col], errors="coerce")
    label = get_column_label(col, subsection_dict, section_dict)

    summary_rows.append({
        "section": label["section"],
        "subsection": label["subsection"],
        "feature": col,
        "feature_type": "reversed" if col.endswith("_rev") else "original",
        "source_feature": col.replace("_rev", "") if col.endswith("_rev") else col,
        "observed_min": values.min(skipna=True),
        "observed_max": values.max(skipna=True),
        "n_missing": values.isna().sum(),
        "pct_missing": values.isna().mean() * 100,
        "value_counts": dict(
            df[col]
            .value_counts(dropna=False)
            .sort_index()
        )
    })

# ------------------------------------------------------------
# Step 3 — Output summary dataframe
# ------------------------------------------------------------
reverse_summary_df = pd.DataFrame(summary_rows)

reverse_summary_df = reverse_summary_df.sort_values(
    by=["source_feature", "feature_type"]
).reset_index(drop=True)

display(reverse_summary_df)

In [ ]:
# ============================================================
# REPLACE ORIGINAL VALUES WITH _rev VALUES
# overwrite original features, then remove *_rev columns
# ============================================================
write_log("Replacing original values with _rev values.", LOG_PATH)

# ------------------------------------------------------------
# Step 1 — Identify all *_rev columns directly from df
# ------------------------------------------------------------
# Use raw dataframe columns as source of truth.
# This avoids dictionary-state issues and guarantees replacement
# and dropping operate on the exact same set of columns.
rev_cols = [col for col in df.columns if col.endswith("_rev")]

if len(rev_cols) == 0:
    write_log("No _rev columns found. Nothing to replace.", LOG_PATH)

else:
    # --------------------------------------------------------
    # Step 2 — Overwrite original values from *_rev copies
    # --------------------------------------------------------
    replaced_features = []

    for rev_col in rev_cols:
        original_col = rev_col.replace("_rev", "")
        # Safety check: original column must exist
        if original_col not in df.columns:
            write_log(f"WARNING: original column missing, skipping replacement for '{rev_col}'", LOG_PATH)
            continue

        # Overwrite original values with corrected values
        df[original_col] = df[rev_col]
        replaced_features.append(original_col)
        write_log(f"Replaced values: {original_col} <- {rev_col}", LOG_PATH)

    # --------------------------------------------------------
    # Step 3 — Drop *_rev columns from dataframe
    # --------------------------------------------------------
    df.drop(columns=rev_cols, inplace=True)
    write_log(f"Dropped {len(rev_cols)} _rev columns from dataframe.", LOG_PATH)

    # --------------------------------------------------------
    # Step 4 — Remove *_rev columns from dictionaries
    # --------------------------------------------------------
    for subsection in subsection_dict:
        subsection_dict[subsection] = [
            c for c in subsection_dict[subsection]
            if not c.endswith("_rev")
        ]

    for section in section_dict:
        section_dict[section] = [
            c for c in section_dict[section]
            if not c.endswith("_rev")
        ]

    subsection_column_names_scales = subsection_column_names_scales[
        ~subsection_column_names_scales["column_name"].str.endswith("_rev")
    ].reset_index(drop=True)

    # --------------------------------------------------------
    # Step 5 — Final logging
    # --------------------------------------------------------
    write_log(f"Replacement complete | overwritten {len(replaced_features)} original features.", LOG_PATH)

# Combined scores: analyze score components, and construct new scores

In [ ]:
# OPTIMISM (LOT-R)

# score is a sum of: 
'''
uncertain_times
go_wrong
optimistic
work_out_myway
good_things
overall
'''
# all questions must be in scale 0-4
# 1. first - must reverse 3 questions; ["go_wrong", "work_out_myway", "good_things"] (make sure the scale here is the correct way, according to literature)
# 2. create optimism_score = sum[optimism_questions]

section = "optimism"
subsection = "optimism"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)
feature_removal_registry.append({
    "columns": subsection_columns,
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

# Quick inspection of created score
display(df[["optimism_score"]].describe())

In [ ]:
# spirituality_score 
# made up of 
'''
spirituality
'''
# scale 0-3
# so maybe dont need to touch. but give a new name?

In [ ]:
# resiliency_score (CD-RISC-2)
'''
adapt
bounce_back
'''

section = "resiliency"
subsection = "resiliency"

# resiliency_score = sum(features) - RE-CONFIRM
sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# ------------------------------------------------------------
# Create composite score
# Default score column name:
#     "<subsection>_score"
# so here:
#     resiliency_score
# ------------------------------------------------------------
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)
feature_removal_registry.append({
    "columns": subsection_columns,
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

# ------------------------------------------------------------
# Quick inspection of created score
# ------------------------------------------------------------
display(df[["resiliency_score"]].describe())

In [ ]:
# coping_strategy_a_score (CISS-21) (maybe give this cluster a better name?)
'''
time_off
snack
friend
buy
special_person
go_out
phone
'''
# scale 1-5 (CONVERT FROM 0-4)
# coping_strategy_a_score = sum(features) - RE-CONFIRM
section = "coping_strategy"
subsection = "coping_strategy_a"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# Create composite score
# Default score column name: "<subsection>_score"
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="coping_strategy_e_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# ------------------------------------------------------------
# Quick inspection of created score
# ------------------------------------------------------------
display(df[["coping_strategy_a_score"]].describe())

In [ ]:
# coping_strategy_e_score (CISS-21) (maybe give this cluster a better name?)
'''
blame
anxious
emotional
very_upset
blame_myself
change_past
inadequacies
'''
# scale 1-5 (CONVERT FROM 0-4)
# coping_strategy_e_score = sum(features) - RE-CONFIRM

section = "coping_strategy"
subsection = "coping_strategy_e"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# ------------------------------------------------------------
# Create composite score
# Default score column name:
#     "<subsection>_score"
# so here:
#     resiliency_score
# ------------------------------------------------------------
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="coping_strategy_e_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# ------------------------------------------------------------
# Quick inspection of created score
# ------------------------------------------------------------
display(df[["coping_strategy_e_score"]].describe())

In [ ]:
# coping_strategy_t_score (CISS-21) (maybe give this cluster a better name?)
'''
solution
similar_problems
course_action
understand
corrective_action
mistakes
analyze
'''
# scale 1-5 (CONVERT FROM 0-4)
# coping_strategy_t_score = sum(features) - RE-CONFIRM
section = "coping_strategy"
subsection = "coping_strategy_t"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# ------------------------------------------------------------
# Create composite score
# Default score column name:
#     "<subsection>_score"
# ------------------------------------------------------------
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="coping_strategy_t_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# ------------------------------------------------------------
# Quick inspection of created score
# ------------------------------------------------------------
display(df[["coping_strategy_t_score"]].describe())


# mark for removal the subsection score components from the df
section_columns = get_section(section_code="coping_strategy", dataframe=df, section_dict=section_dict)
feature_removal_registry.append({
    "columns": section_columns,
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

In [ ]:
# ============================================================
# VISUALIZE COPING STRATEGY SCORES
# ============================================================
write_log("Creating coping strategy visualizations.", log_path=LOG_PATH)

# Section definition
section = "coping_strategy"

# Define subsection score columns and display labels
subsection_score_map = {
    "coping_strategy_t": "task-oriented coping",
    "coping_strategy_e": "emotion-oriented coping",
    "coping_strategy_a": "avoidance coping"
}

# Consistent color palette across ALL figures
subsection_color_map = {
    "coping_strategy_t": "tab:blue",
    "coping_strategy_e": "tab:orange",
    "coping_strategy_a": "tab:green"
}

# ============================================================
# FIGURE 1 — Mean coping strategy scores
# show all participant scores + mean + standard deviation
# ============================================================
mean_score_rows = []

for subsection, label in subsection_score_map.items():
    score_col = f"{subsection}_score"
    if score_col not in df.columns:
        write_log(f"Skipping missing score column: {score_col}")
        continue

    # Store all individual scores
    temp_df = pd.DataFrame({
        "subsection": subsection,
        "label": label,
        "score": df[score_col]
    })

    mean_score_rows.append(temp_df)

# Combine all subsection score rows
mean_scores_df = pd.concat(mean_score_rows, ignore_index=True)

# Summary statistics
summary_df = (mean_scores_df.groupby(["subsection", "label"])["score"].agg(["mean", "std"]).reset_index())

# Plot all scores with jitter + mean ± SD
fig, ax = plt.subplots(figsize=(8, 5))

# Preserve original color mapping
plot_colors = summary_df["subsection"].map(subsection_color_map)

# X positions for categories
x_positions = np.arange(len(summary_df))

label_to_x = {label: x for x, label in enumerate(summary_df["label"])}

# Plot individual participant scores with jitter
for subsection, label in subsection_score_map.items():
    score_col = f"{subsection}_score"
    if score_col not in df.columns:
        continue

    scores = df[score_col].dropna()
    x_center = label_to_x[label]

    # Add horizontal jitter
    jitter = np.random.normal(loc=0,scale=0.06,size=len(scores))

    ax.scatter(np.full(len(scores), x_center) + jitter,scores,color=subsection_color_map[subsection],alpha=0.5,s=30)

# Plot mean and standard deviation
ax.errorbar(
    x=x_positions,
    y=summary_df["mean"],
    yerr=summary_df["std"],
    fmt="_",
    color="black",
    capsize=5,
    markersize=20,
    linewidth=2
)

# Axis formatting
ax.set_xticks(x_positions)
ax.set_xticklabels(summary_df["label"])
ax.set_title("Coping Strategies by Type")
ax.set_xlabel("Coping strategy type")
ax.set_ylabel("Mean score")
plt.xticks(rotation=45, ha="right")

save_plot(
    fig,
    "coping_strategy_mean_scores",
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
)
plt.show()
plt.close()

# ============================================================
# FIGURE 2 — Mean scores of all coping strategy components
# ============================================================

# Get all coping strategy score components
coping_components = sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["score_component"].fillna(False)),
    ["subsection", "column_name"]
].copy()

# Compute mean score per feature
component_rows = []

for _, row in coping_components.iterrows():
    subsection = row["subsection"]
    feature = row["column_name"]

    if feature not in df.columns:
        continue

    feature_mean = pd.to_numeric(df[feature],errors="coerce").mean()

    component_rows.append({
        "subsection": subsection,
        "subsection_label": subsection_score_map.get(subsection, subsection),
        "feature": feature,
        "mean_score": feature_mean
    })

component_means_df = pd.DataFrame(component_rows)

# Assign colors by subsection
bar_colors = component_means_df["subsection"].map(subsection_color_map)

# Plot component means
fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(
    component_means_df["feature"],
    component_means_df["mean_score"],
    color=bar_colors
)

ax.set_title("Mean Scores of Coping Strategy Components, by Strategy Type")
ax.set_xlabel("Feature")
ax.set_ylabel("Mean score")
plt.xticks(rotation=45, ha="right")

# Create legend
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=color) for color in subsection_color_map.values()]
legend_labels = [subsection_score_map[subsection] for subsection in subsection_color_map.keys()]
ax.legend(legend_handles,legend_labels,title="Coping strategy type")
save_plot(
    fig, 
    "coping_strategy_component_means", 
    log_path=LOG_PATH, 
    output_dir=OUTPUT_DIR
)
plt.show()
plt.close()

# ============================================================
# FIGURE 3 — Ternary coping strategy composition plot
# ============================================================
write_log("Creating ternary coping strategy plot.", log_path=LOG_PATH)

# Score columns
score_t = "coping_strategy_t_score"
score_e = "coping_strategy_e_score"
score_a = "coping_strategy_a_score"

# Keep participants with complete scores
ternary_df = df[[score_t, score_e, score_a]].dropna().copy()

# Convert scores to proportions
# Each participant now sums to 1 across the 3 coping styles
row_sums = ternary_df.sum(axis=1)

ternary_df["t"] = ternary_df[score_t] / row_sums
ternary_df["e"] = ternary_df[score_e] / row_sums
ternary_df["a"] = ternary_df[score_a] / row_sums

# Convert ternary coordinates to Cartesian coordinates
x = 0.5 * (2 * ternary_df["e"] + ternary_df["a"])
y = (np.sqrt(3) / 2) * ternary_df["a"]

# Create figure
fig, ax = plt.subplots(figsize=(9, 8))

# Triangle borders
triangle_x = [0, 1, 0.5, 0]
triangle_y = [0, 0, np.sqrt(3)/2, 0]

ax.plot(triangle_x,triangle_y,color="black",linewidth=1.5)

# Add ternary grid lines
grid_levels = np.arange(0.2, 1.0, 0.2)

for g in grid_levels:

    # Constant task-oriented coping
    x1 = 1 - g
    y1 = 0

    x2 = 0.5 * (1 - g)
    y2 = (np.sqrt(3) / 2) * (1 - g)

    ax.plot([x1, x2],[y1, y2],linestyle="--",linewidth=0.7,alpha=0.5,color=subsection_color_map["coping_strategy_t"])

    # Constant emotion-oriented coping
    x1 = g
    y1 = 0

    x2 = 0.5 + 0.5 * g
    y2 = (np.sqrt(3) / 2) * (1 - g)

    ax.plot([x1, x2],[y1, y2],linestyle="--",linewidth=0.7,alpha=0.5,color=subsection_color_map["coping_strategy_e"])

    # Constant avoidance coping
    x1 = 0.5 * g
    y1 = (np.sqrt(3) / 2) * g

    x2 = 1 - 0.5 * g
    y2 = (np.sqrt(3) / 2) * g

    ax.plot([x1, x2],[y1, y2],linestyle="--",linewidth=0.7,alpha=0.5,color=subsection_color_map["coping_strategy_a"])

# Add scatter points
ax.scatter(x,y,alpha=0.65,s=30)

# Axis tick labels (0 → 1)
tick_values = np.arange(0.0, 1.01, 0.2)

for t in tick_values:
    # Bottom axis (task-oriented)
    ax.text(1 - t,-0.035,f"{t:.1f}",fontsize=8,ha="center",color=subsection_color_map["coping_strategy_t"])

    # Right axis (emotion-oriented)
    ax.text(0.5 + 0.5 * t + 0.03,(np.sqrt(3)/2) * (1 - t),f"{t:.1f}",fontsize=8,rotation=-60,va="center",color=subsection_color_map["coping_strategy_e"]    )

    # Left axis (avoidance coping)
    ax.text(0.5 * t - 0.03,(np.sqrt(3)/2) * t,f"{t:.1f}",fontsize=8,rotation=60,va="center",color=subsection_color_map["coping_strategy_a"])

# Axis titles ON triangle edges
# Bottom edge
ax.text(0.5,-0.09,"Task-oriented coping",fontsize=11,ha="center",color=subsection_color_map["coping_strategy_t"])

# Right edge
ax.text(0.82,0.45,"Emotion-oriented coping",fontsize=11,rotation=-60,ha="center",color=subsection_color_map["coping_strategy_e"])

# Left edge
ax.text(0.18,0.45,"Avoidance coping",fontsize=11,rotation=60,ha="center",color=subsection_color_map["coping_strategy_a"])

# Final formatting
ax.set_title("Participant Coping Strategy Profiles")

# Remove outer square completely
ax.axis("off")

# Keep triangle proportions correct
ax.set_aspect("equal")

save_plot(fig, "coping_strategy_ternary", log_path=LOG_PATH, output_dir=OUTPUT_DIR,)
plt.show()
plt.close()

Looking at the densest region: the “typical” participant combines primarily task-oriented coping with a secondary avoidance component and lower emotional coping.

In [ ]:
# perceived_social_support_score (MSPSS)
'''
around
share
fam_tries
fam_emotion
comfort
friend_help
count_on
fam_prblm
frnd_share
feelings
fam_decisions
frnd_prblm
'''
# scale 1-7
# perceived_social_support_score = sum(features) - RE-CONFIRM

section = "support"
subsection = "perceived_social_support"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# ------------------------------------------------------------
# Create composite score
# Default score column name:
#     "<subsection>_score"
# ------------------------------------------------------------
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="coping_strategy_t_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)
feature_removal_registry.append({
    "columns": subsection_columns,
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

# Quick inspection of created score
display(df[["perceived_social_support_score"]].describe())

In [ ]:
# couples_satisfaction_score (CSI-4)
'''
happiness_degree
comfortable
rewarding
satisfied
'''
# scale 0-6
# couples_satisfaction_score = sum(features) - RE-CONFIRM

section = "support"
subsection = "couples_satisfaction"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# Create composite score
# Default score column name: "<subsection>_score"
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="coping_strategy_t_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)
feature_removal_registry.append({
    "columns": subsection_columns,
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

# Quick inspection of created score
display(df[["couples_satisfaction_score"]].describe())

df = create_score_categorization(
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_col="couples_satisfaction_score",
    cutoff=13.5,
    plot=True
)

In [ ]:
# ptsd_b_score (PC-PCL-5) (Re-experiencing)
'''
repeated_img
repeated_dream
hpn_again
upset_remind
physical
'''
# scale 1-5 (convert from 0-4)
# ptsd_b_score = sum(features) - RE-CONFIRM

section = "ptsd"
subsection = "ptsd_b"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# ------------------------------------------------------------
# Create composite score
# Default score column name:
#     "<subsection>_score"
# ------------------------------------------------------------
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="coping_strategy_t_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns_b = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)

# Quick inspection of created score
display(df[["ptsd_b_score"]].describe())

In [ ]:
# ptsd_c_score (PC-PCL-5) (Avoidance and numbing)
'''
avoid_think
avoid_activities
remembering
no_joy
distant
numb
no_future
'''

# scale 1-5 (convert from 0-4)
# ptsd_c_score = sum(features) - RE-CONFIRM


section = "ptsd"
subsection = "ptsd_c"


sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]


score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# ------------------------------------------------------------
# Create composite score
# Default score column name:
#     "<subsection>_score"
# ------------------------------------------------------------
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="coping_strategy_t_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns_c = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)

# Quick inspection of created score
display(df[["ptsd_c_score"]].describe())

In [ ]:
# ptsd_d_score (PC-PCL-5) (Hyperarousal)
'''
trbl_sleep
angry
concentrating
alert
jumpy
'''
# scale 1-5 (convert from 0-4)
# ptsd_d_score = sum(features) - RE-CONFIRM

section = "ptsd"
subsection = "ptsd_d"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# Create composite score
# Default score column name: "<subsection>_score"
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="coping_strategy_t_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns_d = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)

# Quick inspection of created score
display(df[["ptsd_d_score"]].describe())

In [ ]:
# ============================================================
# PCLC SCORE CALCULATION
# ============================================================

write_log("Starting PCLC score calculation.", log_path=LOG_PATH)

# ============================================================
# USER-DEFINED SETTINGS
# ============================================================

# PCLC severity thresholds
PCLC_MINIMAL_MAX = 19
PCLC_SUBTHRESHOLD_MAX = 29

# Automatically derive dependent thresholds
# IMPORTANT: Prevents hardcoded logic elsewhere
PCLC_SUBTHRESHOLD_MIN = (PCLC_MINIMAL_MAX + 1)

PCLC_PROBABLE_MIN = (PCLC_SUBTHRESHOLD_MAX + 1)

# PTSD subsection score columns
PTSD_SCORE_COLUMNS = [
    "ptsd_b_score",
    "ptsd_c_score",
    "ptsd_d_score"
]

# LOG THRESHOLD DEFINITIONS
write_log("PCLC severity thresholds:", log_path=LOG_PATH)

write_log(
    f"minimal/low PTSD symptoms: "
    f"PCLC_score <= {PCLC_MINIMAL_MAX}",
    log_path=LOG_PATH
)

write_log(
    f"sub-threshold PTSD symptoms: "
    f"{PCLC_SUBTHRESHOLD_MIN} <= PCLC_score <= "
    f"{PCLC_SUBTHRESHOLD_MAX}", 
    log_path=LOG_PATH
)

write_log(
    f"probable PTSD: "
    f"PCLC_score >= {PCLC_PROBABLE_MIN}", 
    log_path=LOG_PATH
)

# VALIDATE REQUIRED COLUMNS
write_log("Validating required PTSD subsection score columns.", log_path=LOG_PATH)

missing_cols = [
    col for col in PTSD_SCORE_COLUMNS
    if col not in df.columns
]

if missing_cols:
    raise ValueError(f"Missing PTSD subsection score columns: {missing_cols}")

write_log("All required PTSD subsection score columns found.", log_path=LOG_PATH)

# CREATE NUMERIC PTSD SCORE DATAFRAME
write_log("Creating numeric PTSD score dataframe.", log_path=LOG_PATH)

ptsd_score_df = df[PTSD_SCORE_COLUMNS].apply(pd.to_numeric,errors="coerce")

# IDENTIFY VALID PARTICIPANTS
# IMPORTANT: Participants must have ALL 3 subsection scores

write_log("Identifying participants with complete PTSD subsection scores.",log_path=LOG_PATH)

valid_score_mask = (ptsd_score_df.notna().all(axis=1))

n_valid = valid_score_mask.sum()

write_log(f"Participants with complete PTSD scores: {n_valid}", log_path=LOG_PATH)

# CREATE PCLC SCORE
# IMPORTANT: Only calculate if ALL 3 subsection scores exist

write_log("Creating PCLC_score.", log_path=LOG_PATH)

df["PCLC_score"] = np.nan

df.loc[valid_score_mask,"PCLC_score"] = (ptsd_score_df.loc[valid_score_mask].sum(axis=1))

write_log("PCLC_score created successfully.", log_path=LOG_PATH)

# CREATE PCLC SEVERITY GROUPS

write_log("Assigning PCLC severity groups.", log_path=LOG_PATH)

# Severity group assignment
# IMPORTANT: Uses ONLY configurable threshold variables
df["PCLC_severity_group"] = np.select(
    [
        df["PCLC_score"] <= PCLC_MINIMAL_MAX,

        (
            (df["PCLC_score"] >= PCLC_SUBTHRESHOLD_MIN) &
            (df["PCLC_score"] <= PCLC_SUBTHRESHOLD_MAX)
        ),

        df["PCLC_score"] >= PCLC_PROBABLE_MIN
    ],
    [
        1,
        2,
        3
    ],
    default=np.nan
)

# Convert to nullable integer
df["PCLC_severity_group"] = (df["PCLC_severity_group"].astype("Int64"))

# Human-readable labels
PCLC_SEVERITY_LABELS = {
    1: "minimal/low PTSD symptoms",
    2: "sub-threshold PTSD symptoms",
    3: "probable PTSD"
}

df["PCLC_severity_label"] = (df["PCLC_severity_group"].map(PCLC_SEVERITY_LABELS))

write_log("PCLC severity groups assigned successfully.", log_path=LOG_PATH)

# SUMMARY LOGGING
write_log("PCLC score summary:", log_path=LOG_PATH)

write_log(
    f"Participants with valid PCLC score: "
    f"{df['PCLC_score'].notna().sum()}", 
    log_path=LOG_PATH
)

write_log(
    f"Participants missing PCLC score: "
    f"{df['PCLC_score'].isna().sum()}", 
    log_path=LOG_PATH
)

write_log("PCLC severity distribution:", log_path=LOG_PATH)

for k, v in PCLC_SEVERITY_LABELS.items():

    n_group = (df["PCLC_severity_group"] == k).sum()

    write_log(f"{v}: {n_group}", log_path=LOG_PATH)

write_log("PCLC score calculation completed.", log_path=LOG_PATH)

# mark score component features for removal
feature_removal_registry.append({
    "columns": subsection_columns_b+subsection_columns_c+subsection_columns_d,
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

In [ ]:
# ============================================================
# PCLC VISUALIZATIONS
# ============================================================

write_log("Starting PCLC visualizations.", log_path=LOG_PATH)

# ============================================================
# USER-DEFINED SETTINGS
# ============================================================

# PTSD cluster metadata
PTSD_CLUSTER_INFO = pd.DataFrame({
    "ptsd_cluster": [
        "ptsd_b",
        "ptsd_c",
        "ptsd_d"
    ],
    "cluster_name": [
        "Re-experiencing",
        "Avoidance and numbing",
        "Hyperarousal"
    ],
    "color": [
        "blue",
        "orange",
        "green"
    ]})

# Maximum possible subsection scores
PTSD_CLUSTER_MAX_SCORES = {
    "ptsd_b": 25,
    "ptsd_c": 35,
    "ptsd_d": 25
}

# ============================================================
# CREATE TEMP NORMALIZED DATAFRAME
# ============================================================

write_log("Creating normalized PTSD temporary dataframe.", log_path=LOG_PATH)

temp_PCLC_normalized_df = df[["record_id","ptsd_b_score","ptsd_c_score","ptsd_d_score","PCLC_score","PCLC_severity_group","PCLC_severity_label"]].copy()

# Rename raw subsection columns
temp_PCLC_normalized_df = (
    temp_PCLC_normalized_df.rename(
        columns={
            "ptsd_b_score": "ptsd_b",
            "ptsd_c_score": "ptsd_c",
            "ptsd_d_score": "ptsd_d"
        }))

# Create normalized subsection scores
temp_PCLC_normalized_df["ptsd_b_normalized"] = (temp_PCLC_normalized_df["ptsd_b"] / PTSD_CLUSTER_MAX_SCORES["ptsd_b"])
temp_PCLC_normalized_df["ptsd_c_normalized"] = (temp_PCLC_normalized_df["ptsd_c"] / PTSD_CLUSTER_MAX_SCORES["ptsd_c"])
temp_PCLC_normalized_df["ptsd_d_normalized"] = (temp_PCLC_normalized_df["ptsd_d"] / PTSD_CLUSTER_MAX_SCORES["ptsd_d"])

# Participant usability flag
temp_PCLC_normalized_df["use_participant"] = (temp_PCLC_normalized_df["PCLC_score"].notna())

write_log(
    f"Usable participants: "
    f"{temp_PCLC_normalized_df['use_participant'].sum()}", 
    log_path=LOG_PATH
)

# CREATE ANALYSIS DATAFRAME
analysis_df = temp_PCLC_normalized_df.loc[temp_PCLC_normalized_df["use_participant"]].copy()

# ============================================================
# FIGURE 1 — Participant usability
# ============================================================

write_log("Creating participant usability plot.", log_path=LOG_PATH)

fig, ax = plt.subplots(figsize=(6, 5))

use_counts = (temp_PCLC_normalized_df["use_participant"].value_counts())

bars = ax.bar(use_counts.index.astype(str),use_counts.values)

# Add counts above bars
for bar in bars:

    height = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom"
    )

ax.set_title("Participant usability for PCLC analysis")

ax.set_xlabel("Use participant")
ax.set_ylabel("Participants")

save_plot(fig, "PCLC_participant_usability", output_dir=OUTPUT_DIR, log_path=LOG_PATH)

plt.show()
plt.close()

# ============================================================
# FIGURE 2 — PTSD normalized cluster distributions
# ============================================================

write_log("Creating PTSD normalized cluster distribution plot.", log_path=LOG_PATH)

# Create long-format plotting dataframe
plot_df = pd.DataFrame({
    "Re-experiencing": analysis_df["ptsd_b_normalized"],
    "Avoidance and numbing": analysis_df["ptsd_c_normalized"],
    "Hyperarousal": analysis_df["ptsd_d_normalized"]
})

plot_df = plot_df.melt(
    var_name="cluster",
    value_name="normalized_score"
)

# Cluster colors
cluster_color_map = {
    row["cluster_name"]: row["color"]
    for _, row in PTSD_CLUSTER_INFO.iterrows()
}

# Create figure
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=plot_df,
    x="cluster",
    y="normalized_score",
    palette=cluster_color_map,
    showmeans=True,
    meanprops={
        "marker": "o",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 8
    },
    ax=ax
)

sns.stripplot(
    data=plot_df,
    x="cluster",
    y="normalized_score",
    color="black",
    alpha=0.35,
    jitter=0.2,
    size=4,
    ax=ax
)

ax.set_title("Normalized PTSD symptom cluster distributions")

ax.set_xlabel("PTSD symptom cluster")
ax.set_ylabel("Normalized score")

plt.xticks(rotation=45, ha="right")

save_plot(fig, "PTSD_cluster_normalized_distributions", output_dir=OUTPUT_DIR, log_path=LOG_PATH)

plt.show()
plt.close()

# ============================================================
# FIGURE 3 — Mean PTSD component scores
# ============================================================

write_log("Creating PTSD component mean score plot.", log_path=LOG_PATH)

# Get PTSD component information
# IMPORTANT: subsection column defines PTSD cluster membership
ptsd_components = sections_df.loc[
    (
        sections_df["subsection"]
        .isin(["ptsd_b", "ptsd_c", "ptsd_d"])
    ) &
    (
        sections_df["score_component"]
        .fillna(False)
        .astype(str)
        .str.upper()
        .eq("TRUE")
    ),
    ["subsection", "column_name"]
].copy()

# Compute mean score per PTSD component
component_rows = []

for _, row in ptsd_components.iterrows():

    subsection = row["subsection"]
    feature = row["column_name"]

    if feature not in df.columns:
        continue

    feature_mean = pd.to_numeric(df.loc[analysis_df.index, feature],errors="coerce").mean()

    component_rows.append({
        "ptsd_cluster": subsection,
        "feature": feature,
        "mean_score": feature_mean
    })

component_means_df = pd.DataFrame(component_rows)

# Add cluster names + colors
component_means_df = (
    component_means_df.merge(
        PTSD_CLUSTER_INFO,
        left_on="ptsd_cluster",
        right_on="ptsd_cluster",
        how="left"
    )
)

# Create figure
fig, ax = plt.subplots(figsize=(15, 6))

ax.bar(component_means_df["feature"],component_means_df["mean_score"],color=component_means_df["color"])

ax.set_title("Mean PTSD component scores by symptom cluster")

ax.set_xlabel("PTSD component")
ax.set_ylabel("Mean score")

plt.xticks(rotation=45, ha="right")

# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------
legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=row["color"])
    for _, row in PTSD_CLUSTER_INFO.iterrows()
]

legend_labels = PTSD_CLUSTER_INFO["cluster_name"]

ax.legend(legend_handles,legend_labels,title="PTSD symptom cluster")

save_plot(fig, "PTSD_component_mean_scores", output_dir=OUTPUT_DIR, log_path=LOG_PATH)

plt.show()
plt.close()


# ============================================================
# FIGURE 4 — PCLC score distribution
# ============================================================

write_log("Creating PCLC score distribution plot.", log_path=LOG_PATH)

# Total included participants
n_participants = (analysis_df["PCLC_score"].notna().sum())

# Create figure
fig, ax = plt.subplots(figsize=(10, 6))

analysis_df["PCLC_score"].plot(kind="hist",bins=25,ax=ax)

# Add severity threshold lines
cutoff_lines = [PCLC_SUBTHRESHOLD_MIN,PCLC_PROBABLE_MIN]

for cutoff in cutoff_lines:

    ax.axvline(cutoff,linestyle=":",linewidth=2,color="red")

# Add threshold labels
ymax = ax.get_ylim()[1]

ax.text(PCLC_MINIMAL_MAX / 2,ymax * 0.92,"minimal/low\nPTSD symptoms",ha="center",fontsize=10)

ax.text((PCLC_SUBTHRESHOLD_MIN + PCLC_SUBTHRESHOLD_MAX) / 2,ymax * 0.92,"sub-threshold\nPTSD symptoms",ha="center",fontsize=10)

ax.text((PCLC_PROBABLE_MIN + analysis_df["PCLC_score"].max()) / 2,ymax * 0.92,"probable\nPTSD",ha="center",fontsize=10)

# Add threshold information box
threshold_text = (
    f"Thresholds:\n"
    f"Minimal/Low: ≤ {PCLC_MINIMAL_MAX}\n"
    f"Sub-threshold: "
    f"{PCLC_SUBTHRESHOLD_MIN}–"
    f"{PCLC_SUBTHRESHOLD_MAX}\n"
    f"Probable PTSD: ≥ {PCLC_PROBABLE_MIN}"
)

ax.text(0.98,0.98,threshold_text,transform=ax.transAxes,ha="right",va="top",fontsize=9,bbox=dict(facecolor="white",alpha=0.8))

# Add total N
ax.text(0.02,0.98,f"N = {n_participants}",transform=ax.transAxes,ha="left",va="top",fontsize=10,bbox=dict(facecolor="white",alpha=0.8))

# Formatting
ax.set_title("PCLC score distribution")

ax.set_xlabel("PCLC score")
ax.set_ylabel("Participants")

save_plot(fig, "PCLC_score_distribution", output_dir=OUTPUT_DIR, log_path=LOG_PATH)

plt.show()
plt.close()

write_log("PCLC score distribution plot completed.", log_path=LOG_PATH)

# ============================================================
# FIGURE 4.5 — PCLC score distribution (% participants)
# ============================================================

write_log("Creating percentage-based PCLC score distribution plot.",log_path=LOG_PATH)

# Create figure
fig, ax = plt.subplots(figsize=(10, 6))

# Histogram weights
# Converts counts → percentages
weights = np.ones(len(analysis_df)) * 100 / len(analysis_df)

# Histogram
sns.histplot(
    analysis_df["PCLC_score"],
    bins=25,
    kde=True,
    stat="percent",
    ax=ax
)

# Add severity threshold lines
cutoff_lines = [
    PCLC_SUBTHRESHOLD_MIN,
    PCLC_PROBABLE_MIN
]

for cutoff in cutoff_lines:

    ax.axvline(
        cutoff,
        linestyle="--",
        linewidth=2,
        color="darkblue",
        alpha=0.8
    )

# Add threshold labels
ymax = ax.get_ylim()[1]

ax.text(
    PCLC_SUBTHRESHOLD_MIN + 0.5,
    ymax * 0.95,
    f"Sub-threshold boundary\n({PCLC_SUBTHRESHOLD_MIN})",
    rotation=90,
    va="top",
    ha="left",
    fontsize=10
)

ax.text(
    PCLC_PROBABLE_MIN + 0.5,
    ymax * 0.95,
    f"Probable PTSD boundary\n({PCLC_PROBABLE_MIN})",
    rotation=90,
    va="top",
    ha="left",
    fontsize=10
)

# Summary statistics
score_mean = analysis_df["PCLC_score"].mean()
score_median = analysis_df["PCLC_score"].median()
score_sd = analysis_df["PCLC_score"].std()

summary_text = (
    f"N = {n_participants}\n"
    f"Mean = {score_mean:.1f}\n"
    f"Median = {score_median:.1f}\n"
    f"SD = {score_sd:.1f}"
)

ax.text(
    0.98,
    0.97,
    summary_text,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=11,
    bbox=dict(
        facecolor="lightgrey",
        alpha=0.5,
        edgecolor="none"
    )
)

# Formatting
ax.set_title("Distribution of PTSD Symptom Severity (PCL-C Total Score)")

ax.set_xlabel("PCL-C Total Score (1to5 scoring)")

ax.set_ylabel("Participants (%)")

save_plot(fig,"PCLC_score_distribution_percent",output_dir=OUTPUT_DIR,log_path=LOG_PATH)

plt.show()
plt.close()

write_log("Percentage-based PCLC score distribution plot completed.",log_path=LOG_PATH)

# ============================================================
# FIGURE 5 — PCLC severity group distribution
# ============================================================

write_log("Creating PCLC severity group distribution plot.", log_path=LOG_PATH)

severity_counts = (analysis_df["PCLC_severity_label"].value_counts())

severity_order = [
    "minimal/low PTSD symptoms",
    "sub-threshold PTSD symptoms",
    "probable PTSD"
]

severity_counts = severity_counts.reindex(severity_order)

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(severity_counts.index,severity_counts.values)

# Add participant counts
for bar in bars:

    height = bar.get_height()

    ax.text(bar.get_x() + bar.get_width()/2,height,f"N={int(height)}",ha="center",va="bottom")

ax.set_title("PCLC severity group distribution")

ax.set_xlabel("PCLC severity group")
ax.set_ylabel("Participants")

plt.xticks(rotation=45, ha="right")

save_plot(fig, "PCLC_severity_distribution", output_dir=OUTPUT_DIR, log_path=LOG_PATH)

plt.show()
plt.close()

# ============================================================
# FIGURE 5.5 — PCLC severity group distribution (%)
# ============================================================

write_log("Creating percentage-based PCLC severity group distribution.",log_path=LOG_PATH)

# Compute counts + percentages
severity_counts = (analysis_df["PCLC_severity_label"].value_counts())

severity_order = ["minimal/low PTSD symptoms","sub-threshold PTSD symptoms","probable PTSD"]

severity_counts = severity_counts.reindex(severity_order)

severity_percent = (severity_counts / severity_counts.sum()) * 100

# Create display labels
display_labels = [
    f"Minimal / Low (<{PCLC_SUBTHRESHOLD_MIN})",
    (
        f"Sub-threshold "
        f"({PCLC_SUBTHRESHOLD_MIN}–"
        f"{PCLC_SUBTHRESHOLD_MAX})"
    ),
    f"Probable PTSD ({PCLC_PROBABLE_MIN}+)"
]

# Consistent color palette
severity_colors = ["tab:blue","tab:orange","tab:green"]

# Create figure
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(display_labels,severity_percent.values,color=severity_colors)

# Add percentage + n labels above bars
for bar, pct, n in zip(bars,severity_percent.values,severity_counts.values):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{pct:.1f}%, n={n}",
        ha="center",
        va="bottom",
        fontsize=10
    )

# Formatting
ax.set_title("PTSD Severity Category Distribution")

ax.set_xlabel("PTSD Severity Category")

ax.set_ylabel("Participants (%)")

plt.xticks(rotation=15,ha="right")

save_plot(
    fig,
    "PCLC_severity_distribution_percent", 
    output_dir=OUTPUT_DIR, 
    log_path=LOG_PATH
)

plt.show()
plt.close()

write_log("Percentage-based PCLC severity group distribution completed.",log_path=LOG_PATH)


# ============================================================
# FIGURE 6 — PTSD symptom cluster ternary plot
# ============================================================

write_log("Creating PTSD symptom cluster ternary plot.", log_path=LOG_PATH)

# Ternary proportions
row_sums = (
    analysis_df["ptsd_b_normalized"] +
    analysis_df["ptsd_c_normalized"] +
    analysis_df["ptsd_d_normalized"]
)

analysis_df["b_prop"] = (analysis_df["ptsd_b_normalized"] / row_sums)

analysis_df["c_prop"] = (analysis_df["ptsd_c_normalized"] / row_sums)

analysis_df["d_prop"] = (analysis_df["ptsd_d_normalized"] / row_sums)

# Convert ternary coordinates
x = 0.5 * (2 * analysis_df["c_prop"] + analysis_df["d_prop"])
y = (np.sqrt(3) / 2) * analysis_df["d_prop"]

# Create figure
fig, ax = plt.subplots(figsize=(9, 8))

triangle_x = [0, 1, 0.5, 0]
triangle_y = [0, 0, np.sqrt(3)/2, 0]

ax.plot(
    triangle_x,
    triangle_y,
    color="black",
    linewidth=1.5
)

# Grid lines
grid_levels = np.arange(0.2, 1.0, 0.2)

for g in grid_levels:

    ax.plot(
        [1 - g, 0.5 * (1 - g)],
        [0, (np.sqrt(3)/2) * (1 - g)],
        linestyle="--",
        linewidth=0.7,
        alpha=0.5,
        color="blue"
    )

    ax.plot(
        [g, 0.5 + 0.5 * g],
        [0, (np.sqrt(3)/2) * (1 - g)],
        linestyle="--",
        linewidth=0.7,
        alpha=0.5,
        color="orange"
    )

    ax.plot(
        [0.5 * g, 1 - 0.5 * g],
        [(np.sqrt(3)/2) * g,
         (np.sqrt(3)/2) * g],
        linestyle="--",
        linewidth=0.7,
        alpha=0.5,
        color="green"
    )

# Scatter points
ax.scatter(x,y,alpha=0.65,s=30)

# Axis labels
ax.text(0.5,-0.09,"Re-experiencing",fontsize=11,ha="center",color="blue")

ax.text(0.82,0.45,"Avoidance and numbing",fontsize=11,rotation=-60,ha="center",color="orange")

ax.text(0.18,0.45,"Hyperarousal",fontsize=11,rotation=60,ha="center",color="green")

# Tick labels
tick_values = np.arange(0.0, 1.01, 0.2)

for t in tick_values:

    ax.text(1 - t,-0.035,f"{t:.1f}",fontsize=8,ha="center",color="blue")

    ax.text(0.5 + 0.5 * t + 0.03,(np.sqrt(3)/2) * (1 - t),f"{t:.1f}",fontsize=8,rotation=-60,va="center",color="orange")

    ax.text(0.5 * t - 0.03,(np.sqrt(3)/2) * t,f"{t:.1f}",fontsize=8,rotation=60,va="center",color="green")

# Final formatting
ax.set_title("Participant PTSD symptom cluster profiles")

ax.axis("off")

ax.set_aspect("equal")

save_plot(fig, "PTSD_ternary_plot", output_dir=OUTPUT_DIR, log_path=LOG_PATH)

plt.show()
plt.close()

write_log("PCLC visualizations completed.", log_path=LOG_PATH)

In [ ]:
# anxiety_score (GAD-2)
'''
on_edge
control
'''
# scale 0-3
# anxiety_score = sum(features)

section = "outcomes"
subsection = "anxiety"


sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# Create composite score
# Default score column name: "<subsection>_score"
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="coping_strategy_t_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)
feature_removal_registry.append({
    "columns": subsection_columns,
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

# Quick inspection of created score
display(df[["anxiety_score"]].describe())

# add categorization using the cutoff
df = create_score_categorization(
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_col="anxiety_score",
    cutoff=3,
    plot=True
)

In [ ]:
# depression_score (PHQ2)
'''
little_intrest
down
'''
# scale 0-3
# depression_score = sum(features)

section = "outcomes"
subsection = "depression"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# Create composite score
# Default score column name: "<subsection>_score"
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
   #score_column_name="ptg_relating_others_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)
feature_removal_registry.append({
    "columns": subsection_columns,
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

# Quick inspection of created score
display(df[["depression_score"]].describe())

# add categorization using the cutoff
df = create_score_categorization(
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_col="depression_score",
    cutoff=3,
    plot=True
)

In [ ]:
# ptg_relating_others_score
# original: post_traumatic_growth_1_score (maybe give this cluster a better name?) (PTGI)
'''
closeness
emotions
compassion
relationships
wonderful_ppl
needing
'''
# scale 0-5
# ptg_relating_others_score = sum(features)

section = "post_traumatic_growth"
subsection = "post_traumatic_growth_1"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# Create composite score
# Default score column name: "<subsection>_score"
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="ptg_relating_others_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns_1 = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)

# Quick inspection of created score
display(df[["post_traumatic_growth_1_score"]].describe())

In [ ]:
# ptg_new_possibilities_score
# original: post_traumatic_growth_2_score (maybe give this cluster a better name?) (PTGI)
'''
new_interests
count_on_people
new_path
better
opportunities
changing
'''
# scale 0-5
# ptg_new_possibilities_score = sum(features)

section = "post_traumatic_growth"
subsection = "post_traumatic_growth_2"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# Create composite score
# Default score column name: "<subsection>_score"
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="ptg_new_possibilities_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns_2 = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)

# Quick inspection of created score
display(df[["post_traumatic_growth_2_score"]].describe())

In [ ]:
# ptg_personal_strength_score
# original: post_traumatic_growth_3_score (maybe give this cluster a better name?) (PTGI)
'''
self_reliance
handle
accept
stronger
'''
# scale 0-5
# ptg_personal_strength_score = sum(features)

section = "post_traumatic_growth"
subsection = "post_traumatic_growth_3"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# Create composite score
# Default score column name: "<subsection>_score"
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="ptg_personal_strength_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns_3 = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)

# Quick inspection of created score
display(df[["post_traumatic_growth_3_score"]].describe())

In [ ]:
# ptg_spiritual_score
# original: post_traumatic_growth_4_score (maybe give this cluster a better name?) (PTGI)
'''
spiritual
religious
'''
# scale 0-5
# ptg_spiritual_score = sum(features)

section = "post_traumatic_growth"
subsection = "post_traumatic_growth_4"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# Create composite score
# Default score column name: "<subsection>_score"
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="ptg_spiritual_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns_4 = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)

# Quick inspection of created score
display(df[["post_traumatic_growth_4_score"]].describe())

In [ ]:
# ptg_life_appreciation_score
# original: post_traumatic_growth_5_score (maybe give this cluster a better name?) (PTGI)
'''
priorities
appreciation
appreciate
'''
# scale 0-5
# ptg_life_appreciation_score = sum(features)

section = "post_traumatic_growth"
subsection = "post_traumatic_growth_5"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# Create composite score
# Default score column name: "<subsection>_score"
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="ptg_life_appreciation_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns_5 = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)

# Quick inspection of created score
display(df[["post_traumatic_growth_5_score"]].describe())

In [ ]:
# ============================================================
# PTGI SCORE CALCULATION
# ============================================================

write_log("Starting PTGI score calculation.", log_path=LOG_PATH)

# ============================================================
# USER-DEFINED SETTINGS
# ============================================================
# PTGI growth thresholds
PTGI_MINIMAL_MAX = 35
PTGI_MODERATE_MAX = 70

# Automatically derive dependent thresholds
PTGI_MODERATE_MIN = (PTGI_MINIMAL_MAX + 1)

PTGI_HIGH_MIN = (PTGI_MODERATE_MAX + 1)

# PTGI subsection score columns
# IMPORTANT: These are assumed to already be subsection scores
PTGI_SCORE_COLUMNS = [
    "post_traumatic_growth_1_score",
    "post_traumatic_growth_2_score",
    "post_traumatic_growth_3_score",
    "post_traumatic_growth_4_score",
    "post_traumatic_growth_5_score"
]

# LOG THRESHOLD DEFINITIONS

write_log("PTGI growth thresholds:", log_path=LOG_PATH)

write_log(f"minimal/none: PTGI total score <= {PTGI_MINIMAL_MAX}", log_path=LOG_PATH)

write_log(f"moderate: {PTGI_MODERATE_MIN} <= PTGI total score <= {PTGI_MODERATE_MAX}", log_path=LOG_PATH)

write_log(f"high: PTGI total score >= {PTGI_HIGH_MIN}", log_path=LOG_PATH)

# VALIDATE REQUIRED COLUMNS
write_log("Validating required PTGI score columns.", log_path=LOG_PATH)

missing_cols = [
    col for col in PTGI_SCORE_COLUMNS
    if col not in df.columns
]

if missing_cols:
    raise ValueError(f"Missing PTGI score columns: {missing_cols}")

write_log("All required PTGI score columns found.", log_path=LOG_PATH)

# CREATE NUMERIC PTGI SCORE DATAFRAME
write_log("Creating numeric PTGI score dataframe.", log_path=LOG_PATH)

ptgi_score_df = df[PTGI_SCORE_COLUMNS].apply(pd.to_numeric,errors="coerce")

# IDENTIFY VALID PARTICIPANTS
# IMPORTANT: Participants must have ALL 5 subsection scores

write_log("Identifying participants with complete PTGI scores.", log_path=LOG_PATH)

valid_score_mask = (ptgi_score_df.notna().all(axis=1))

n_valid = valid_score_mask.sum()

write_log(f"Participants with complete PTGI scores: {n_valid}", log_path=LOG_PATH)

# CREATE PTGI TOTAL SCORE
# IMPORTANT: Only calculate if ALL subsection scores exist

write_log("Creating ptg_i_score.", log_path=LOG_PATH)

df["ptg_i_score"] = np.nan

df.loc[valid_score_mask,"ptg_i_score"] = (ptgi_score_df.loc[valid_score_mask].sum(axis=1))

write_log("ptg_i_score created successfully.", log_path=LOG_PATH)

# CREATE PTGI GROWTH GROUPS
write_log("Assigning PTGI growth groups.", log_path=LOG_PATH)

# Growth group assignment
# IMPORTANT: Uses ONLY configurable threshold variables
df["ptg_i_growth_group"] = np.select(
    [
        df["ptg_i_score"] <= PTGI_MINIMAL_MAX,

        (
            (df["ptg_i_score"] >= PTGI_MODERATE_MIN) &
            (df["ptg_i_score"] <= PTGI_MODERATE_MAX)
        ),

        df["ptg_i_score"] >= PTGI_HIGH_MIN
    ],
    [
        1,
        2,
        3
    ],
    default=np.nan
)

# Convert to nullable integer
df["ptg_i_growth_group"] = (df["ptg_i_growth_group"].astype("Int64"))

# Human-readable labels
PTGI_GROWTH_LABELS = {
    1: "minimal/none",
    2: "moderate",
    3: "high"
}

df["ptg_i_growth_label"] = (df["ptg_i_growth_group"].map(PTGI_GROWTH_LABELS))

write_log("PTGI growth groups assigned successfully.", log_path=LOG_PATH)

# SUMMARY LOGGING
write_log("PTGI score summary:", log_path=LOG_PATH)
write_log(f"Participants with valid PTGI score: {df['ptg_i_score'].notna().sum()}", log_path=LOG_PATH)
write_log(f"Participants missing PTGI score: {df['ptg_i_score'].isna().sum()}", log_path=LOG_PATH)
write_log("PTGI growth distribution:", log_path=LOG_PATH)

for k, v in PTGI_GROWTH_LABELS.items():

    n_group = (df["ptg_i_growth_group"] == k).sum()

    write_log(f"{v}: {n_group}", log_path=LOG_PATH)

write_log("PTGI score calculation completed.", log_path=LOG_PATH)

# mark score component features, for removal
feature_removal_registry.append({
    "columns": subsection_columns_5+subsection_columns_4+subsection_columns_3+subsection_columns_2+subsection_columns_1,
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

In [ ]:
# ============================================================
# PTGI VISUALIZATIONS
# ============================================================
write_log("Starting PTGI visualizations.", log_path=LOG_PATH)

# ============================================================
# USER-DEFINED SETTINGS
# ============================================================

# ------------------------------------------------------------
# PTGI cluster metadata
# Distinguishes:
# - subsection identifier
# - score column
# - normalized column
# ------------------------------------------------------------
PTGI_CLUSTER_INFO = pd.DataFrame({

    "ptgi_cluster": [
        "post_traumatic_growth_1",
        "post_traumatic_growth_2",
        "post_traumatic_growth_3",
        "post_traumatic_growth_4",
        "post_traumatic_growth_5"
    ],

    "score_column": [
        "post_traumatic_growth_1_score",
        "post_traumatic_growth_2_score",
        "post_traumatic_growth_3_score",
        "post_traumatic_growth_4_score",
        "post_traumatic_growth_5_score"
    ],

    "normalized_cluster": [
        "post_traumatic_growth_1_normalized",
        "post_traumatic_growth_2_normalized",
        "post_traumatic_growth_3_normalized",
        "post_traumatic_growth_4_normalized",
        "post_traumatic_growth_5_normalized"
    ],

    "cluster_name": [
        "Relating others",
        "New possibilities",
        "Personal strength",
        "Spiritual",
        "Life appreciation"
    ],

    "color": [
        "cyan",
        "teal",
        "orange",
        "magenta",
        "gold"
    ]
})

# Maximum possible subsection scores
# IMPORTANT: Adjust if scoring system changes
PTGI_CLUSTER_MAX_SCORES = {
    "post_traumatic_growth_1": 30,
    "post_traumatic_growth_2": 30,
    "post_traumatic_growth_3": 20,
    "post_traumatic_growth_4": 10,
    "post_traumatic_growth_5": 15
}

# VALIDATE REQUIRED COLUMNS

required_cols = (["record_id"] + PTGI_CLUSTER_INFO["score_column"].tolist() + ["ptg_i_score","ptg_i_growth_group","ptg_i_growth_label"])

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols: 
    raise ValueError(f"Missing required columns: {missing_cols}")

# CREATE TEMP NORMALIZED DATAFRAME
write_log("Creating normalized PTGI temporary dataframe.", log_path=LOG_PATH)

temp_PTGI_normalized_df = df[required_cols].copy()

# Create normalized subsection scores
for _, row in PTGI_CLUSTER_INFO.iterrows():

    cluster = row["ptgi_cluster"]
    score_col = row["score_column"]
    normalized_col = row["normalized_cluster"]

    max_score = PTGI_CLUSTER_MAX_SCORES[cluster]

    temp_PTGI_normalized_df[normalized_col] = (temp_PTGI_normalized_df[score_col] / max_score)

# Participant usability flag
# IMPORTANT: Participants must have valid PTGI total score
temp_PTGI_normalized_df["use_participant"] = (temp_PTGI_normalized_df["ptg_i_score"].notna())

write_log(f"Usable participants: {temp_PTGI_normalized_df['use_participant'].sum()}", log_path=LOG_PATH)

# CREATE ANALYSIS DATAFRAME
analysis_df = temp_PTGI_normalized_df.loc[temp_PTGI_normalized_df["use_participant"]].copy()

# ============================================================
# FIGURE 1 — Participant usability
# ============================================================
write_log("Creating participant usability plot.", log_path=LOG_PATH)
fig, ax = plt.subplots(figsize=(6, 5))

use_counts = (temp_PTGI_normalized_df["use_participant"].value_counts())

bars = ax.bar(use_counts.index.astype(str),use_counts.values)

# Add participant counts
for bar in bars:

    height = bar.get_height()

    ax.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom"
    )

ax.set_title("Participant usability for PTGI analysis")
ax.set_xlabel("Use participant")
ax.set_ylabel("Participants")

save_plot(fig, "PTGI_participant_usability", output_dir=OUTPUT_DIR, log_path=LOG_PATH)

plt.show()
plt.close()

# ============================================================
# FIGURE 2 — PTGI normalized distributions
# ============================================================

write_log("Creating PTGI normalized cluster distribution plot.", log_path=LOG_PATH)

# Create long-format plotting dataframe
plot_df = pd.DataFrame()

for _, row in PTGI_CLUSTER_INFO.iterrows():

    temp_df = pd.DataFrame({
        "cluster": row["cluster_name"],
        "normalized_score": analysis_df[
            row["normalized_cluster"]
        ]
    })

    plot_df = pd.concat([plot_df, temp_df],axis=0)

# Cluster color map
cluster_color_map = {
    row["cluster_name"]: row["color"]
    for _, row in PTGI_CLUSTER_INFO.iterrows()
}

# Create figure
fig, ax = plt.subplots(figsize=(12, 6))

sns.boxplot(
    data=plot_df,
    x="cluster",
    y="normalized_score",
    palette=cluster_color_map,
    showmeans=True,
    meanprops={
        "marker": "o",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 8
    },
    ax=ax
)

sns.stripplot(
    data=plot_df,
    x="cluster",
    y="normalized_score",
    color="black",
    alpha=0.25,
    jitter=0.2,
    size=3,
    ax=ax
)

ax.set_title("Normalized post-traumatic growth distributions")

ax.set_xlabel("PTGI domain")
ax.set_ylabel("Normalized score")

plt.xticks(rotation=45, ha="right")

save_plot(
    fig,
    "PTGI_cluster_normalized_distributions",
    output_dir=OUTPUT_DIR,
    log_path=LOG_PATH
)

plt.show()
plt.close()

# ============================================================
# FIGURE 3 — Mean PTGI component scores
# ============================================================
write_log("Creating PTGI component mean score plot.", log_path=LOG_PATH)

# Get PTGI component information
ptgi_components = sections_df.loc[
    (
        sections_df["subsection"]
        .isin(
            PTGI_CLUSTER_INFO["ptgi_cluster"]
        )
    ) &
    (
        sections_df["score_component"]
        .fillna(False)
        .astype(str)
        .str.upper()
        .eq("TRUE")
    ),
    ["subsection", "column_name"]
].copy()

# Compute mean score per PTGI component
component_rows = []

for _, row in ptgi_components.iterrows():

    subsection = row["subsection"]
    feature = row["column_name"]

    if feature not in df.columns:
        continue

    feature_mean = pd.to_numeric(df.loc[analysis_df.index,feature],errors="coerce").mean()

    component_rows.append({
        "ptgi_cluster": subsection,
        "feature": feature,
        "mean_score": feature_mean
    })

component_means_df = pd.DataFrame(component_rows)

# Add cluster names + colors
component_means_df = (
    component_means_df.merge(
        PTGI_CLUSTER_INFO,
        left_on="ptgi_cluster",
        right_on="ptgi_cluster",
        how="left"
    )
)

# Create figure
fig, ax = plt.subplots(figsize=(16, 6))

ax.bar(component_means_df["feature"],component_means_df["mean_score"],color=component_means_df["color"])

ax.set_title("Mean PTGI component scores by growth domain")

ax.set_xlabel("PTGI component")
ax.set_ylabel("Mean score")

plt.xticks(rotation=45, ha="right")

# Legend
legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=row["color"])
    for _, row in PTGI_CLUSTER_INFO.iterrows()
]

legend_labels = PTGI_CLUSTER_INFO["cluster_name"]
ax.legend(legend_handles,legend_labels,title="PTGI domain")
save_plot(fig, "PTGI_component_mean_scores", output_dir=OUTPUT_DIR, log_path=LOG_PATH)
plt.show()
plt.close()

# ============================================================
# FIGURE 4 — PTGI total score distribution
# ============================================================
write_log("Creating PTGI score distribution plot.", log_path=LOG_PATH)

n_participants = (analysis_df["ptg_i_score"].notna().sum())

fig, ax = plt.subplots(figsize=(10, 6))

analysis_df["ptg_i_score"].plot(kind="hist",bins=25,ax=ax)

# Threshold lines
cutoff_lines = [ PTGI_MODERATE_MIN,PTGI_HIGH_MIN]

for cutoff in cutoff_lines:
    ax.axvline(
        cutoff,
        linestyle=":",
        linewidth=2,
        color="red"
    )

# Threshold labels
ymax = ax.get_ylim()[1]

ax.text(PTGI_MINIMAL_MAX / 2,ymax * 0.92,"minimal/none",ha="center")

ax.text((PTGI_MODERATE_MIN + PTGI_MODERATE_MAX) / 2, ymax * 0.92, "moderate", ha="center")

ax.text((PTGI_HIGH_MIN + analysis_df["ptg_i_score"].max()) / 2, ymax * 0.92, "high", ha="center")

# Summary statistics
summary_text = (
    f"N = {n_participants}\n"
    f"Mean = "
    f"{analysis_df['ptg_i_score'].mean():.1f}\n"
    f"Median = "
    f"{analysis_df['ptg_i_score'].median():.1f}\n"
    f"SD = "
    f"{analysis_df['ptg_i_score'].std():.1f}"
)

ax.text(0.98,0.98,summary_text,transform=ax.transAxes,ha="right",va="top",fontsize=10,bbox=dict(facecolor="white",alpha=0.8))

ax.set_title("PTGI total score distribution")
ax.set_xlabel("PTGI total score")
ax.set_ylabel("Participants")

save_plot(fig, "PTGI_total_score_distribution", output_dir=OUTPUT_DIR, log_path=LOG_PATH)
plt.show()
plt.close()

# ============================================================
# FIGURE 5 — PTGI growth category distribution
# ============================================================
write_log("Creating PTGI growth category distribution plot.", log_path=LOG_PATH)

growth_counts = (analysis_df["ptg_i_growth_label"].value_counts())

growth_order = ["minimal/none","moderate","high"]

growth_counts = growth_counts.reindex(growth_order)

growth_percent = (growth_counts / growth_counts.sum()) * 100

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(growth_counts.index,growth_percent.values)

# Add percentages + n
for bar, pct, n in zip(bars,growth_percent.values,growth_counts.values):

    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{pct:.1f}%, n={n}",
        ha="center",
        va="bottom"
    )

ax.set_title("PTGI growth category distribution")
ax.set_xlabel("PTGI growth category")
ax.set_ylabel("Participants (%)")
plt.xticks(rotation=45, ha="right")

save_plot(fig, "PTGI_growth_distribution", output_dir=OUTPUT_DIR, log_path=LOG_PATH)
plt.show()
plt.close()

# ============================================================
# FIGURE 6 — Participant PTGI profiles
# PARALLEL COORDINATES PLOT
# ============================================================

write_log("Creating PTGI participant profile plot.", log_path=LOG_PATH)

# Create plotting dataframe
parallel_df = analysis_df[PTGI_CLUSTER_INFO["normalized_cluster"].tolist()].copy()

# Rename columns to display labels
rename_dict = dict(zip(PTGI_CLUSTER_INFO["normalized_cluster"],PTGI_CLUSTER_INFO["cluster_name"]))

parallel_df = parallel_df.rename(columns=rename_dict)

# Create figure
fig, ax = plt.subplots(figsize=(14, 7))

# Plot each participant
for _, row in parallel_df.iterrows():

    ax.plot(parallel_df.columns,row.values,alpha=0.08,color="black")

# Add mean profile
mean_profile = parallel_df.mean()

ax.plot(parallel_df.columns,mean_profile.values,linewidth=4,color="red",label="Mean profile")

# Formatting
ax.set_title("Participant post-traumatic growth profiles")
ax.set_xlabel("PTGI domain")
ax.set_ylabel("Normalized score")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
ax.legend()

save_plot(fig, "PTGI_parallel_profiles", output_dir=OUTPUT_DIR, log_path=LOG_PATH)
plt.show()
plt.close()

write_log("PTGI visualizations completed.", log_path=LOG_PATH)

In [ ]:
ptgi_pca = plot_pca(
    dataframe=analysis_df,
    feature_cols=[
        "post_traumatic_growth_1_normalized",
        "post_traumatic_growth_2_normalized",
        "post_traumatic_growth_3_normalized",
        "post_traumatic_growth_4_normalized",
        "post_traumatic_growth_5_normalized"
    ],
    color_col="ptg_i_growth_label",
    title="PTGI participant PCA by growth group"
)

In [ ]:
# ACE_score
# adverse childhood experiences

'''
ace_swear
ace_alcohol_drug
'''

# make sure again how to calculate it (it says scale is 0-1, how come if there are 2 questions?)

section = "adverse_childhood_experiences"
subsection = "adverse_childhood_experiences"

sections_df.loc[
    (sections_df["section"] == section) &
    (sections_df["subsection"] == subsection),
    ["section", "subsection", "column_name", "score_component"]
]

score_qc = analyze_score_inputs(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    plot=True
)

In [ ]:
# Create composite score
# Default score column name: "<subsection>_score"
df = create_composite_score(
    sections_df=sections_df,
    section=section,
    subsection=subsection,
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_method="sum",      # or "mean"
    #score_column_name="ptg_life_appreciation_score",
    min_valid_fraction=1,   # score will only be calculated if the participant has all of the components
    plot=True
)

# remove score components from the df
subsection_columns = get_subsection(subsection_code=subsection, dataframe=df, subsection_dict=subsection_dict)
feature_removal_registry.append({
    "columns": subsection_columns,
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

# Quick inspection of created score
display(df[["adverse_childhood_experiences_score"]].describe())

df = create_score_categorization(
    dataframe=df,
    score_col="adverse_childhood_experiences_score",
    cutoff=1,
    plot=True,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR
)

In [ ]:
# add column: after_iran_war, based on the feature days_since_oct7, cutoff is 875 days (feb 28, 2026)
df = create_score_categorization(
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    score_col="days_since_oct7",
    cutoff=875,
    output_col="after_iran_war",
    plot=True
)

In [ ]:
# psych history and drug history - add new features: *_total, *_any
# what they represent? 
# - baseline vulnerability
# - chronic psychiatric burden
# - prior mental-health history
# - pre-existing risk

# ignore psych_hx___98, make score of psych_hx
'''
psych_hx___1
psych_hx___2
psych_hx___3
psych_hx___4
psych_hx___5
psych_hx___6
psych_hx___7
psych_hx___8
psych_hx___99
psych_hx___98 - ignore
si_before
psych_med
psych_drug___1
psych_drug___2
psych_drug___3
psych_drug___4
psych_drug___5
psych_drug___6
psych_drug___7
psych_drug___8
psych_drug___9
psych_drug___10
psych_drug___11
psych_drug___99
other_psychmed
'''

psych_hx_cols = [
    "psych_hx___1",
    "psych_hx___2",
    "psych_hx___3",
    "psych_hx___4",
    "psych_hx___5",
    "psych_hx___6",
    "psych_hx___7",
    "psych_hx___8",
    "psych_hx___99"
]

df = create_binary_summary_features(
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    columns=psych_hx_cols,
    plot=True
)

psych_drug_cols = [
    "psych_drug___1",
    "psych_drug___2",
    "psych_drug___3",
    "psych_drug___4",
    "psych_drug___5",
    "psych_drug___6",
    "psych_drug___7",
    "psych_drug___8",
    "psych_drug___9",
    "psych_drug___10",
    "psych_drug___11",
    "psych_drug___99"
]

df = create_binary_summary_features(
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    columns=psych_drug_cols,
    plot=True
)

# mark score component feature for removal
feature_removal_registry.append({
    "columns": psych_hx_cols+psych_drug_cols+["psych_hx___98","psych_drug___98"],
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

In [ ]:
# psych history and drug use since 7/10
# these scores represent: 
# - incident psychopathology
# - post-trauma deterioration
# - possible trauma-associated emergence
# - psychiatric escalation during wartime

'''
new_psychdx___1
new_psychdx___2
new_psychdx___3
new_psychdx___4
new_psychdx___5
new_psychdx___6
new_psychdx___7
new_psychdx___8
new_psychdx___99 
new_psychdx___98 - check this
si
sa
psych_med_war___1
psych_med_war___2
psych_med_war___3
psych_med_war___4
psych_med_war___5
psych_med_war___6
psych_med_war___7
psych_med_war___8
psych_med_war___9
psych_med_war___10
psych_med_war___11
psych_med_war___99
other_psychmed_2
'''

# convert the columns into one-hot encoding (free text to 1) and give a total score
# keep the original, too? maybe not for models
new_psychdx_cols = [
    "new_psychdx___1",
    "new_psychdx___2",
    "new_psychdx___3",
    "new_psychdx___4",
    "new_psychdx___5",
    "new_psychdx___6",
    "new_psychdx___7",
    "new_psychdx___8",
    "new_psychdx___99"
]

df = create_binary_summary_features(
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    columns=new_psychdx_cols,
    plot=True
)


psych_med_war_cols = [
    "psych_med_war___1",
    "psych_med_war___2",
    "psych_med_war___3",
    "psych_med_war___4",
    "psych_med_war___5",
    "psych_med_war___6",
    "psych_med_war___7",
    "psych_med_war___8",
    "psych_med_war___9",
    "psych_med_war___10",
    "psych_med_war___11",
    "psych_med_war___99"
]

df = create_binary_summary_features(
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    columns=psych_med_war_cols,
    plot=True
)

feature_removal_registry.append({
    "columns": new_psychdx_cols+psych_med_war_cols+["new_psychdx___98","psych_med_war___98"],
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

In [ ]:
# ============================================================
# CREATE TEMPORAL PSYCHIATRIC BURDEN FEATURES
# ============================================================
# psych and drug use derived features: fraction and difference between before war and during war.
# Difference captures:
# - chronicity
# - baseline vulnerability
# - accumulated psychiatric burden

# Fraction captures:
# - temporal shift
# - emergence pattern
# - relative war-associated contribution


write_log("Creating temporal psychiatric burden features.", log_path=LOG_PATH)

# Convert to numeric
psych_total = pd.to_numeric(df["psych_hx_total"],errors="coerce")

war_total = pd.to_numeric(df["new_psychdx_total"],errors="coerce")

# ============================================================
# FEATURE 1 — Pre-war psychiatric burden
# ============================================================

df["psych_hx_war_difference"] = (psych_total - war_total)

# ============================================================
# FEATURE 2 — Fraction of burden emerging during war
# ============================================================
df["psych_hx_war_fraction"] = np.nan

# Avoid division by zero
# IMPORTANT: If psych_total == 0; fraction remains NaN
valid_fraction_mask = (
    psych_total.notna() &
    war_total.notna() &
    (psych_total > 0)
)

df.loc[valid_fraction_mask,"psych_hx_war_fraction"] = (war_total[valid_fraction_mask] / psych_total[valid_fraction_mask])

# Logging
write_log("Temporal psychiatric burden features created.", log_path=LOG_PATH)

write_log(
    f"Valid psych_hx_war_fraction rows: "
    f"{valid_fraction_mask.sum()}",
    log_path=LOG_PATH
)

write_log(
    f"Missing psych_hx_war_fraction rows: "
    f"{df['psych_hx_war_fraction'].isna().sum()}",
    log_path=LOG_PATH
)

# VISUALIZATIONS

write_log("Creating temporal psychiatric burden visualizations.", log_path=LOG_PATH)

# ============================================================
# FIGURE 1 — Distribution of psych_hx_war_difference
# ============================================================
difference_plot_df = pd.to_numeric(df["psych_hx_war_difference"], errors="coerce")

# Remove inf values
difference_plot_df = difference_plot_df.replace([np.inf, -np.inf], np.nan)
difference_plot_df = (difference_plot_df.dropna().astype(float))
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(
    x=difference_plot_df.to_numpy(),
    bins=30,
    kde=True,
    ax=ax
)

# Add zero reference line
ax.axvline(0,linestyle=":",linewidth=2,color="red")

# Add summary statistics
stats_text = (
    f"N = {len(difference_plot_df)}\n"
    f"Mean = {difference_plot_df.mean():.2f}\n"
    f"Median = {difference_plot_df.median():.2f}\n"
    f"SD = {difference_plot_df.std():.2f}\n"
    f"Min = {difference_plot_df.min():.2f}\n"
    f"Max = {difference_plot_df.max():.2f}"
)

ax.text(0.98,0.98,stats_text,transform=ax.transAxes,ha="right",va="top",fontsize=10,bbox=dict(facecolor="white",alpha=0.85))

# Labels
ax.set_title("Distribution of Pre-War Psychiatric Burden")
ax.set_xlabel("psych_hx_total - new_psychdx_total")
ax.set_ylabel("Participants")

save_plot(
    fig,
    "psych_hx_war_difference_distribution",
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
)

plt.show()
plt.close()

# ============================================================
# FIGURE 2 — Distribution of psych_hx_war_fraction
# ============================================================
fraction_plot_df = pd.to_numeric(df["psych_hx_war_fraction"],errors="coerce")

# Remove inf values
fraction_plot_df = fraction_plot_df.replace([np.inf, -np.inf],np.nan)
fraction_plot_df = (fraction_plot_df.dropna().astype(float))
fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(x=fraction_plot_df.to_numpy(),bins=30,kde=True,ax=ax)

# Add boundary reference lines
for x in [0, 0.5, 1]:

    ax.axvline(x,linestyle=":",linewidth=1.5)

# Add summary statistics
stats_text = (
    f"N = {len(fraction_plot_df)}\n"
    f"Mean = {fraction_plot_df.mean():.2f}\n"
    f"Median = {fraction_plot_df.median():.2f}\n"
    f"SD = {fraction_plot_df.std():.2f}\n"
    f"Min = {fraction_plot_df.min():.2f}\n"
    f"Max = {fraction_plot_df.max():.2f}"
)

ax.text(0.98,0.98,stats_text,transform=ax.transAxes,ha="right",va="top",fontsize=10,bbox=dict(facecolor="white",alpha=0.85))

# Labels
ax.set_title("Fraction of Psychiatric Burden Emerging During War")
ax.set_xlabel("new_psychdx_total / psych_hx_total")
ax.set_ylabel("Participants")
ax.set_xlim(0, 1)

save_plot(
    fig,
    "psych_hx_war_fraction_distribution",
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
)
plt.show()
plt.close()

# ============================================================
# FIGURE 3 — Relationship between total burden and war burden
# ============================================================
scatter_df = pd.DataFrame({
    "psych_total": psych_total,
    "war_total": war_total
})

scatter_df = (scatter_df.replace([np.inf, -np.inf], np.nan).dropna())
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(scatter_df["psych_total"],scatter_df["war_total"],alpha=0.7)

# Add identity line
# IMPORTANT: new_psychdx_total should theoretically not exceed psych_hx_total
max_axis = max(scatter_df["psych_total"].max(),scatter_df["war_total"].max())

ax.plot([0, max_axis],[0, max_axis],linestyle="--",linewidth=2,color="red")

# Labels
ax.set_title("Lifetime vs Wartime Psychiatric Burden")
ax.set_xlabel("Lifetime psychiatric burden (psych_hx_total)")
ax.set_ylabel("Wartime psychiatric burden (new_psychdx_total)")

save_plot(
    fig,
    "psych_hx_vs_new_psychdx_scatter",
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
)
plt.show()
plt.close()

write_log("Temporal psychiatric burden visualizations completed.", log_path=LOG_PATH)

In [ ]:
# ============================================================
# COUNT PARTICIPANTS WHERE: psych_hx_total < new_psychdx_total
# ============================================================
# Convert to numeric
psych_total = pd.to_numeric(df["psych_hx_total"],errors="coerce")
war_total = pd.to_numeric(df["new_psychdx_total"],errors="coerce")

# Identify impossible/inconsistent cases
invalid_mask = (psych_total < war_total)
n_invalid = invalid_mask.sum()

write_log(
    f"Participants with "
    f"psych_hx_total < new_psychdx_total: "
    f"{n_invalid}", 
    log_path=LOG_PATH
)

display(df.loc[invalid_mask,["record_id","psych_hx_total","new_psychdx_total"]])

In [ ]:
# ============================================================
# HANDLE TEMPORALLY INCONSISTENT PSYCHIATRIC HISTORY PARTICIPANTS
# ============================================================
write_log("Checking for temporal inconsistencies in psychiatric history variables.", log_path=LOG_PATH)

# DEFINE RELEVANT FEATURES
# Features to invalidate
# IMPORTANT: These features assume psych_hx_total >= new_psychdx_total
invalidate_features = [
    "psych_hx_total",
    "psych_hx_any",
    "psych_hx_war_difference",
    "psych_hx_war_fraction"
]

# Validate required columns exist
missing_cols = [col for col in invalidate_features + ["psych_hx_total","new_psychdx_total"] if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# ==============================================
# IDENTIFY TEMPORALLY INCONSISTENT PARTICIPANTS
# ==============================================
# Convert to numeric
psych_total = pd.to_numeric(df["psych_hx_total"],errors="coerce")
war_total = pd.to_numeric(df["new_psychdx_total"],errors="coerce")

# Define inconsistency:
# wartime burden exceeds lifetime burden
inconsistency_mask = (psych_total < war_total)
n_inconsistent = inconsistency_mask.sum()

# CREATE QC FLAG
# 1 = temporally inconsistent
# 0 = logically consistent
df["qc_psych_history_temporal_inconsistency"] = (inconsistency_mask).astype("Int64")

write_log(
    f"Participants with temporal inconsistency: "
    f"{n_inconsistent}", 
    log_path=LOG_PATH,
)

# Get problematic participant IDs
problematic_ids = (df.loc[inconsistency_mask,"record_id"].astype(str).tolist())

write_log("Problematic record_ids:", log_path=LOG_PATH)

write_log(", ".join(problematic_ids), log_path=LOG_PATH)

write_log("Decision:", log_path=LOG_PATH)

write_log(
    "For participants where "
    "new_psychdx_total > psych_hx_total, "
    "baseline psychiatric history variables are "
    "considered unreliable due to temporal "
    "inconsistency in questionnaire interpretation.",
    log_path=LOG_PATH
)

write_log(
    "The wartime psychiatric burden variables "
    "(new_psychdx_*) are retained.",
    log_path=LOG_PATH
)

write_log(
    "The following features will be invalidated "
    "(set to NaN) for these participants:",
    log_path=LOG_PATH
)

write_log(", ".join(invalidate_features), log_path=LOG_PATH)

# ============================================================
# INVALIDATE FEATURES
# ============================================================
# Set inconsistent baseline-derived features to NaN
df.loc[inconsistency_mask,invalidate_features] = np.nan

write_log("Feature invalidation completed.", log_path=LOG_PATH)

for feature in invalidate_features:

    n_missing = (df[feature].isna().sum())

    write_log(
        f"{feature}: "
        f"{n_missing} missing values",
    log_path=LOG_PATH)

write_log("Temporal psychiatric history QC completed.", log_path=LOG_PATH)

In [ ]:
# psych_history_family_score? CONTINUE FROM HERE!!! OR MAYBE DONT BOTHER...... dont know how important this is...
'''
psych_fam_hx
psych_fam_dx___1
psych_fam_dx___2
psych_fam_dx___3
psych_fam_dx___4
psych_fam_dx___5
sibling_dx
parent_dx
aunt_unc_dx
cous_dx
grandparent_dx
'''
psych_fam_dx_cols = [
    "psych_fam_dx___1",
    "psych_fam_dx___2",
    "psych_fam_dx___3",
    "psych_fam_dx___4",
    "psych_fam_dx___5"
]

df = create_binary_summary_features(
    dataframe=df,
    log_path=LOG_PATH,
    output_dir=OUTPUT_DIR,
    columns=psych_fam_dx_cols,
    plot=True
)

# mark score component features for removal
feature_removal_registry.append({
    "columns": psych_fam_dx_cols,
    "removal_reason": f"{section} score components",
    "removal_stage": "score_construction"
})

In [ ]:
# health_rate is baseline health score - maybe dont need to change, but to know that. scale 1-5

In [ ]:
# ============================================================
# CREATE MEDIA CONSUMPTION PERCENT CHANGE
# ============================================================

write_log("Creating media_reserve_perchange feature.", log_path=LOG_PATH)

# Convert features to numeric
baseline = pd.to_numeric(df["media_2wk_clean"],errors="coerce")

test = pd.to_numeric(df["media_reserv_clean"],errors="coerce")

# Initialize output column
# IMPORTANT: Avoid division-by-zero errors
df["media_reserve_perchange"] = np.nan

# Only calculate where:
# - baseline exists
# - test exists
# - baseline is not zero
valid_mask = (
    baseline.notna() &
    test.notna() &
    (baseline != 0)
)

# Percent change formula
df.loc[
    valid_mask,
    "media_reserve_perchange"
] = (
    (
        test[valid_mask] -
        baseline[valid_mask]
    )
    / baseline[valid_mask]
) * 100


write_log("media_reserve_perchange created successfully.", log_path=LOG_PATH)

write_log(
    f"Valid percent-change rows: "
    f"{valid_mask.sum()}",
    log_path=LOG_PATH
)

write_log(
    f"Missing percent-change rows: "
    f"{df['media_reserve_perchange'].isna().sum()}",
    log_path=LOG_PATH
)

In [ ]:
# ============================================================
# MEDIA CHANGE DIRECTION SUMMARY
# ============================================================

write_log("Calculating media consumption change direction.", log_path=LOG_PATH)

# Count increase / decrease / no-change
n_increased = (df["media_reserve_perchange"] > 0).sum()

n_decreased = (df["media_reserve_perchange"] < 0).sum()

n_no_change = (df["media_reserve_perchange"] == 0).sum()

n_valid = (df["media_reserve_perchange"].notna().sum())

# Percentages
pct_increased = (n_increased / n_valid) * 100

pct_decreased = (n_decreased / n_valid) * 100

pct_no_change = (n_no_change / n_valid) * 100

write_log(f"Increased: {n_increased} "
    f"({pct_increased:.1f}%)",
    log_path=LOG_PATH)

write_log(f"Decreased: {n_decreased} "
    f"({pct_decreased:.1f}%)",
    log_path=LOG_PATH)

write_log(f"No change: {n_no_change} "
    f"({pct_no_change:.1f}%)",
    log_path=LOG_PATH)

# VISUALIZATION
direction_df = pd.DataFrame({"change_direction": ["Increased","Decreased","No change"],
                             "n_participants": [n_increased,n_decreased,n_no_change],
                             "percent": [pct_increased,pct_decreased,pct_no_change]})

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(direction_df["change_direction"],direction_df["percent"])

# Add labels above bars
for bar, pct, n in zip(bars,direction_df["percent"],direction_df["n_participants"]):

    ax.text(bar.get_x() + bar.get_width()/2,bar.get_height(),f"{pct:.1f}%, N={n}",ha="center",va="bottom")

# Labels
ax.set_title("Direction of Media Consumption Change")
ax.set_xlabel("Change direction")
ax.set_ylabel("Participants (%)")

save_plot(
    fig,
    "media_change_direction",
    output_dir=OUTPUT_DIR,
    log_path=LOG_PATH
)
plt.show()
plt.close()

In [ ]:
# ============================================================
# PCA VISUALIZATION — Mental health profile structure
# ============================================================
write_log("Creating PCA visualizations for mental health scores.", log_path=LOG_PATH)

# FEATURES FOR PCA
pca_features = [
    "depression_score",
    "anxiety_score",
    "couples_satisfaction_score",
    "optimism_score",
    "resiliency_score"
]

# VALIDATE FEATURES
missing_cols = [
    col for col in pca_features
    if col not in df.columns
]

if missing_cols:
    raise ValueError(f"Missing PCA features: {missing_cols}")

# CREATE PCA DATAFRAME

pca_df = df[pca_features].apply(pd.to_numeric,errors="coerce")

# Keep only complete rows
valid_mask = pca_df.notna().all(axis=1)
pca_df = pca_df.loc[valid_mask].copy()
write_log(f"PCA participants retained: {len(pca_df)}", log_path=LOG_PATH)

# STANDARDIZE FEATURES
# IMPORTANT: PCA is variance-sensitive
scaler = StandardScaler()
scaled_data = scaler.fit_transform(pca_df)

# RUN PCA
pca = PCA(n_components=2)
pca_components = pca.fit_transform(scaled_data)
explained_variance = (pca.explained_variance_ratio_ * 100)

write_log(
    f"PC1 variance explained: {explained_variance[0]:.2f}%", log_path=LOG_PATH)

write_log(f"PC2 variance explained: {explained_variance[1]:.2f}%", log_path=LOG_PATH)

# CREATE MASTER PCA PLOTTING DATAFRAME

plot_df = pd.DataFrame({"PC1": pca_components[:, 0],"PC2": pca_components[:, 1]})

# Add metadata columns
metadata_cols = ["after_iran_war", "PCLC_severity_label", "fraction_in_reserve"]

for col in metadata_cols:
    if col in df.columns:
        plot_df[col] = (df.loc[pca_df.index,col].values)

# ============================================================
# FIGURE 1 — PCA colored by after_iran_war
# ============================================================
write_log("Creating PCA colored by after_iran_war.",log_path=LOG_PATH)
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=plot_df,
    x="PC1",
    y="PC2",
    hue="after_iran_war",
    alpha=0.75,
    ax=ax
)

ax.set_title("PCA of Mental Health Scores\nColored by After Iran War")
ax.set_xlabel(f"PC1 ({explained_variance[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1f}% variance)")

save_plot(
    fig,
    "mental_health_pca_after_iran_war",
    output_dir=OUTPUT_DIR,
    log_path=LOG_PATH
)
plt.show()
plt.close()

# ============================================================
# FIGURE 2 — PCA colored by PTSD severity
# ============================================================
write_log("Creating PCA colored by PTSD severity.", log_path=LOG_PATH)

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=plot_df,
    x="PC1",
    y="PC2",
    hue="PCLC_severity_label",
    alpha=0.75,
    ax=ax
)

ax.set_title("PCA of Mental Health Scores\nColored by PTSD Severity")
ax.set_xlabel(f"PC1 ({explained_variance[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1f}% variance)")

save_plot(
    fig,
    "mental_health_pca_ptsd_severity",
    output_dir=OUTPUT_DIR,
    log_path=LOG_PATH
)
plt.show()
plt.close()

# ============================================================
# FIGURE 3 — PCA colored by fraction_in_reserve
# ============================================================
write_log("Creating PCA colored by fraction_in_reserve.", log_path=LOG_PATH)
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    plot_df["PC1"],
    plot_df["PC2"],
    c=plot_df["fraction_in_reserve"],
    alpha=0.75
)

# Add colorbar
cbar = plt.colorbar(scatter,ax=ax)

cbar.set_label("Fraction in reserve")

# Labels
ax.set_title("PCA of Mental Health Scores\nColored by Fraction in Reserve")
ax.set_xlabel(f"PC1 ({explained_variance[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1f}% variance)")

save_plot(
    fig,
    "mental_health_pca_fraction_in_reserve",
    output_dir=OUTPUT_DIR,
    log_path=LOG_PATH
)
plt.show()
plt.close()

write_log("Mental health PCA visualizations completed.", log_path=LOG_PATH)

In [ ]:
# ============================================================
# PCA VISUALIZATION — Mixed mental health profile structure
# ============================================================
write_log("Creating PCA visualizations using mixed continuous + binary mental health features.", log_path=LOG_PATH)

# FEATURES FOR PCA
# Includes both:
# - binary categorization features
# - continuous psychometric scores
pca_features = [
    "depression_score_categorization",
    "anxiety_score_categorization",
    "couples_satisfaction_score_categorization",
    "optimism_score",
    "resiliency_score"
]

# VALIDATE FEATURES
missing_cols = [
    col for col in pca_features
    if col not in df.columns
]

if missing_cols:
    raise ValueError(f"Missing PCA features: {missing_cols}")

# CREATE PCA DATAFRAME
pca_df = df[pca_features].apply(pd.to_numeric,errors="coerce")

# Keep only complete rows
valid_mask = pca_df.notna().all(axis=1)
pca_df = pca_df.loc[valid_mask].copy()

write_log(f"PCA participants retained: {len(pca_df)}", log_path=LOG_PATH)

# STANDARDIZE FEATURES
# IMPORTANT: Binary + continuous features exist on different scales
scaler = StandardScaler()
scaled_data = scaler.fit_transform(pca_df)

# RUN PCA
pca = PCA(n_components=2)
pca_components = pca.fit_transform(scaled_data)

explained_variance = (pca.explained_variance_ratio_ * 100)

write_log(f"PC1 variance explained: {explained_variance[0]:.2f}%", log_path=LOG_PATH)

write_log(f"PC2 variance explained: {explained_variance[1]:.2f}%", log_path=LOG_PATH)

# CREATE MASTER PCA PLOTTING DATAFRAME
plot_df = pd.DataFrame({
    "PC1": pca_components[:, 0],
    "PC2": pca_components[:, 1]
})

# Add metadata columns
metadata_cols = [
    "after_iran_war",
    "PCLC_severity_label",
    "fraction_in_reserve"
]

for col in metadata_cols:
    if col in df.columns:
        plot_df[col] = (df.loc[pca_df.index,col].values)

# ============================================================
# FIGURE 1 — PCA colored by after_iran_war
# ============================================================
write_log("Creating PCA colored by after_iran_war.", log_path=LOG_PATH)
fig, ax = plt.subplots(figsize=(8, 6))

sns.scatterplot(data=plot_df,x="PC1",y="PC2",hue="after_iran_war",alpha=0.75,ax=ax)

ax.set_title(
    "PCA of Mental Health Profiles\n"
    "Colored by After Iran War Status"
)
ax.set_xlabel(f"PC1 ({explained_variance[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1f}% variance)")

save_plot(
    fig,
    "mental_health_pca_after_iran_war_binary",
    output_dir=OUTPUT_DIR,
    log_path=LOG_PATH
)
plt.show()
plt.close()

# ============================================================
# FIGURE 2 — PCA colored by PTSD severity
# ============================================================
write_log("Creating PCA colored by PTSD severity.",log_path=LOG_PATH
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=plot_df,x="PC1",y="PC2",hue="PCLC_severity_label",alpha=0.75,ax=ax)

ax.set_title(
    "PCA of Mental Health Profiles\n"
    "Colored by PTSD Severity"
)

ax.set_xlabel(f"PC1 ({explained_variance[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1f}% variance)")

save_plot(
    fig,
    "mental_health_pca_ptsd_severity_binary",
    output_dir=OUTPUT_DIR,
    log_path=LOG_PATH
)
plt.show()
plt.close()

# ============================================================
# FIGURE 3 — PCA colored by fraction_in_reserve
# ============================================================
write_log("Creating PCA colored by fraction_in_reserve.", log_path=LOG_PATH)
fig, ax = plt.subplots(figsize=(8, 6))

scatter = ax.scatter(plot_df["PC1"],plot_df["PC2"],c=plot_df["fraction_in_reserve"],alpha=0.75)

# Add colorbar
cbar = plt.colorbar(scatter,ax=ax)
cbar.set_label("Fraction in reserve")

# Labels
ax.set_title(
    "PCA of Mental Health Profiles\n"
    "Colored by Fraction in Reserve"
)

ax.set_xlabel(f"PC1 ({explained_variance[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1f}% variance)")

save_plot(
    fig,
    "mental_health_pca_fraction_in_reserve_binary",
    output_dir=OUTPUT_DIR,
    log_path=LOG_PATH
)
plt.show()
plt.close()

write_log("Mental health PCA visualizations completed.", log_path=LOG_PATH)

# ============================================================
# FIGURE 4 — PCA colored by ptg_i_growth_label
# ============================================================
write_log("Creating PCA colored by ptg_i_growth_label.",log_path=LOG_PATH)

# Add PTGI labels to plot dataframe
if "ptg_i_growth_label" not in plot_df.columns:
    if "ptg_i_growth_label" not in df.columns:
        raise ValueError("ptg_i_growth_label not found in df.")

    plot_df["ptg_i_growth_label"] = (df.loc[pca_df.index,"ptg_i_growth_label"].values)

# Create figure
fig, ax = plt.subplots(figsize=(8, 6))

sns.scatterplot(data=plot_df,x="PC1",y="PC2",hue="ptg_i_growth_label",alpha=0.75,ax=ax)

# Labels
ax.set_title(
    "PCA of Mental Health Profiles\n"
    "Colored by PTGI Growth Category"
)
ax.set_xlabel(f"PC1 ({explained_variance[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1f}% variance)")

save_plot(
    fig,
    "mental_health_pca_ptg_i_growth_label",
    output_dir=OUTPUT_DIR,
    log_path=LOG_PATH
)
plt.show()
plt.close()

write_log("PTGI PCA visualization completed.", log_path=LOG_PATH)

In [ ]:
# TODO: move all of the columns that i now have _clean for (i think mainly as result of the data prep stage). maybe go over all columns to 
# make sure i am not missing anything

In [ ]:
# remove all of the unnecessary features
for removal_job in feature_removal_registry:

    (df, removed_feature_values_df, removal_log_df) = audit_and_remove_features(
        dataframe=df,
        columns=removal_job["columns"],
        removal_reason=removal_job["removal_reason"],
        log_path=LOG_PATH,
        removed_feature_values_df=removed_feature_values_df,
        removal_log_df=removal_log_df,
        feature_removal_audit_path=FEATURE_REMOVAL_AUDIT_PATH,
        override_main_df=True,
        removal_stage=removal_job["removal_stage"]
    )

In [ ]:
# ============================================================
# SAVE DATAFRAME as engineered up to this point
# ============================================================
output_file = (OUTPUT_DIR / "data_after_preprocess_with_scores.xlsx")
df.to_excel(output_file, index=False)
write_log(f"Saved dataframe to: {output_file}", log_path=LOG_PATH)

removal_log_df.to_excel(FEATURE_REMOVAL_LOG_PATH, index=False)
write_log(f"Saved removal_log_df to: {FEATURE_REMOVAL_LOG_PATH}", log_path=LOG_PATH)